# rate_design batch: run infrastructure for the 66 launchable runs (v2.2)

**What this notebook does.** The rate_design batch is pre-registered
(`z-ethan/rate_design/exports/u80_batch_spec.csv`, 54 rows; `u81_run_schedules.csv`, the 48
offered credit paths; gates GE1/GE2/GH1/GH2/GX1/GA1/GA2 in `z-ethan/rate_design/methods.md`
v2.1 + v2.2). This notebook TRANSCRIBES that spec into ReEDS inputs — every emission is
asserted back against u80/u81 — and applies exactly one registered rule of its own: the
selection of the 12 p25/p75 anchor worlds (v2.2) by the frozen smr100 quantile rule.

1. **48 new ITC-arm cost worlds** (27 envelope + 7 boundary-depth + 14 hybrid runs; each
   run has its own world). The MC is re-run with the production generator's own code —
   ported verbatim at build time from `smr100_case_export.ipynb` with content asserts, QA-0
   pinned to `mc_perdraw.npz` — and the u80 draw indices are selected from the materialized
   10,000 draws. Each world gets the standard file set: two plantchar files,
   `financials_tech_mc_*`, `construction_times_mc_*`, and a `foreign_experience_rd_*` file
   at its own drawn `u` (via `_generate_foreign_experience.py --u`, the itcfb precedent).
2. **48 per-run incentives files** on the exact minus-probe row template
   (`build_itcfb_minus.py`): Nuclear-SMR rows only, one per build year, `safe_harbor=0`,
   `itc_tax_equity_penalty=0.1` (ReEDS monetizes at 0.9 internally — the registered
   monetized-parity convention), `itc_frac` = the u81 `rate_on_world` at 3 decimals.
   The hybrid runs' post-window cap (0.60, or 0.50 for the `cb50` probes) is already baked
   into u81's offers.
3. **12 anchor-densification worlds** (`smr100_{sched}_p25` / `_p75`, v2.2): selected here
   by the frozen smr100 rule `argsort(score)[ceil(q·(N−1))]` at q = 0.25 / 0.75 on the
   program-NPV ranking, asserted distinct from the 18 frozen anchors and from every ITC-arm
   world, registered in `u82_anchor_spec.csv` and `exports/smr100/selected_draws_p25p75.csv`
   (the frozen `selected_draws.csv` is never rewritten). Their columns are the smr100 anchor
   pattern verbatim (mandate ON, learning OFF, no-nuclear-ITC baseline, endyear 2050) with
   their own plantchar/financials/construction-time files; no foreign-experience file
   (learning OFF) and no incentives file (baseline).
4. **`cases_nuclearlearning_ratedesign.csv`** — 66 columns on the itcfbm 31-row template,
   ordered envelope → boundary → hybrid → anchor → **horizon last** (Ethan 09-02: the
   standard-horizon runs return results first if the extension path fails). No reserve
   (v2.2: the constraint is one batch, so every slot is live).
5. **Horizon plumbing (endyear 2055).** The 6 horizon runs are reruns of the
   `smr100_{sched}_p50` anchor columns (their worlds ARE the frozen p50 draws — asserted)
   with `endyear=2055`, yearset extended `..._2050_2053_2055`, and the mandate held flat at
   its 2050 level via explicit `nuclear_cap_trajectory_{token}_smr_ext.csv` files (the
   registered design; pre-extended so forecast.py never has to parse the GAMS `*t` header).
   `futurefiles.csv` gets ignore-rows for the nuclear-learning inputs_case files so
   forecast.py's raise-on-missing check passes; everything else is projected by its
   existing rows (`plantcharout.csv` constant = the drawn 2050 cost held flat — "the world
   after 2050 looks like 2050").

**Run-arm convention (logged as a build-time clarification, not a spec change):** the 48
envelope/boundary/hybrid runs use the **fbC convention** — `GSw_NuclearLearning=1` with the
run world's own drawn parameters, no mandate, credit as the only instrument — because the
delivery certificate the batch extends (fbC full-headline delivery + the r03 bracket) was
earned on that arm, and methods.md v2.1 registers "the feed-back tier convention". The 6
horizon runs are mandate-dual reruns (mandate ON, no-nuclear-ITC baseline, learning OFF),
exactly the anchor configuration.

Frozen inputs (byte-identity asserted at the end): u80/u81, the smr100 registrations and
input files, the itcfbm casefile template, `incentives_obbba_nonuclearitc.csv`.


In [1]:
import json
import os
import zlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# --- Paths: anchored to this notebook's folder, never the launch directory ---
NB_DIR = Path.cwd() if Path.cwd().name == "mc" else Path("z-ethan/mc").resolve()
assert (NB_DIR / "pris_loader.py").exists(), f"notebook folder not found from {Path.cwd()}"
REPO_ROOT = NB_DIR.parent.parent               # the ReEDS-nuclear-learning fork
EXPORTS = NB_DIR / "exports" / "ratedesign"
FIGURES = NB_DIR / "figures"
for d in (EXPORTS, FIGURES):
    d.mkdir(parents=True, exist_ok=True)

import sys
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

# --- Reproducibility: one master seed, independent named streams. The world streams use
# the COMPANION notebook's names ("world/{schedule}") so the drawn worlds here are
# bit-identical to mc_cost_trajectories.ipynb's (QA-0 asserts this); streams unique to
# this notebook are prefixed "smr100/". ---
MASTER_SEED = 20260715

def rng_stream(name):
    """An independent, reproducible random generator tied to a label."""
    return np.random.default_rng(np.random.SeedSequence((MASTER_SEED, zlib.crc32(name.encode()))))

# --- Analysis switches ---
COPULA_SET = "moderate"                # within-tech lr <-> boak correlation (as companions)
RANKING_FUNCTIONAL = "discounted_schedule_weighted_npv"  # issue-8 default weights, NPV cost object (2026-08-03)
# NOTE: deliberately differs from mc_cost_trajectories' financed-CAPEX ranking - the mc_ and
# smr100 case families' percentile labels are NOT on a common scale (see README).
N_DRAWS = int(os.environ.get("RATEDESIGN_DRAWS", 10000))  # per schedule; env override for test runs

# --- Model frame (identical to the companion notebooks) ---
YEARS = np.arange(2024, 2051)
T = len(YEARS)
def yi(y):
    """Index of calendar year y in the YEARS grid."""
    return int(np.where(YEARS == y)[0][0])

ANCHOR = 2030
N_BOAK_UNITS = 2.0
OMEGA = 1.0 / 3.0
CES_RHO_GRID = np.array([-1.0, 0.0, 1.0])   # symmetric since 2026-08-06 (companion S1)
X_SPILL_MAX = 0.30     # cross-tech spillover fractions drawn U(0, X_SPILL_MAX) per direction
TECH = {
    "large": {"unit_gw": 1.000, "lr_lo": 0.03, "lr_hi": 0.12,
              "boak_lo": 5250.0, "boak_hi": 7750.0},
    "smr":   {"unit_gw": 0.300, "lr_lo": 0.03, "lr_hi": 0.16,
              "boak_lo": 5500.0, "boak_hi": 10000.0},
}
UNIT_FOREIGN_GW = 1.0
U_GRID = np.linspace(0.0, 1.0, 21)

print(f"notebook dir : {NB_DIR}")
print(f"switches     : copula={COPULA_SET}, ranking={RANKING_FUNCTIONAL}, draws/schedule={N_DRAWS}")
print("deployment   : 100% SMR (large rides the loser channel: intl spillover + x * SMR program)")

notebook dir : C:\Users\ethan\code\research\ReEDS-nuclear-learning\z-ethan\mc
switches     : copula=moderate, ranking=discounted_schedule_weighted_npv, draws/schedule=10000
deployment   : 100% SMR (large rides the loser channel: intl spillover + x * SMR program)


In [2]:
# --- Frozen-artifact guard: the batch spec and every reused registration must be
# byte-identical after this notebook runs. construction_schedules_mc.csv is in the list
# deliberately: the Export-2 cell re-writes it from the same deterministic code, so
# equality doubles as a determinism check (the step4 pattern).
import hashlib

def _sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

RD_EXPORTS = REPO_ROOT / "z-ethan" / "rate_design" / "exports"
GUARDED_FILES = [
    RD_EXPORTS / "u80_batch_spec.csv",
    RD_EXPORTS / "u81_run_schedules.csv",
    RD_EXPORTS / "u80_batch_spec_v21.csv",        # the v2.1 audit record (frozen 09-04)
    RD_EXPORTS / "u81_run_schedules_v21.csv",
    NB_DIR / "exports" / "smr100" / "selected_draws.csv",
    REPO_ROOT / "cases_nuclearlearning_smr100.csv",
    REPO_ROOT / "cases_nuclearlearning_itcfbm.csv",
    REPO_ROOT / "inputs" / "financials" / "incentives_obbba_nonuclearitc.csv",
    REPO_ROOT / "inputs" / "financials" / "construction_schedules_mc.csv",
    REPO_ROOT / "inputs" / "plant_characteristics" / "nuclear_mc_smr100_eo_p50.csv",
]
GUARD_SHA = {p: _sha(p) for p in GUARDED_FILES}
print(f"snapshotted {len(GUARD_SHA)} frozen artifacts (verified unchanged in QA-R7)")


snapshotted 10 frozen artifacts (verified unchanged in QA-R7)


## Ported foundations

Verbatim from `smr100_case_export.ipynb` (itself porting `mc_cost_trajectories.ipynb`
S2–S7): schedules, PRIS experience stocks, the OCC engine, the INL duration model, the
ReEDS financing replication, the copula draw, and the program-NPV ranking. Same worlds,
same seed streams; QA-0 below pins them to `mc_perdraw.npz`.


In [3]:
US_GW_2024 = 97.0
OFFSET_GW = 3.0

US_SCHEDULES_CSV = REPO_ROOT / "inputs" / "nuclear_learning" / "US_SCHEDULES.csv"
_sched_df = pd.read_csv(US_SCHEDULES_CSV, comment="#", index_col="year")
TOKEN_NAME = {"eia_aeo_high": "EIA 2026 AEO high", "abou_jaoude": "Abou-Jaoude mod",
              "iaea_high": "IAEA high", "mckinsey": "McKinsey GEP 2025",
              "cop28": "COP28 tripling pledge", "eo2025": "2025 EO"}
assert set(_sched_df.columns) == set(TOKEN_NAME), "US_SCHEDULES.csv columns changed"
assert list(_sched_df.index) == list(YEARS), "US_SCHEDULES.csv year grid changed"
US_SCHEDULES = {TOKEN_NAME[tok]: _sched_df[tok].to_numpy(float) for tok in TOKEN_NAME}
SCHED_ORDER = list(US_SCHEDULES)
SCEN_TOKEN = {v: k for k, v in TOKEN_NAME.items()}

REGION_MILESTONES = {
    "CA":  (12.7, 12.7, 12.7, 18.8, 23.0, 30.0, 63.7),
    "SAM": (5.1, 5, 5, 8, 11, 8, 20),
    "WEU": (95.4, 86, 92, 81, 114, 68, 136),
    "EEU": (53.6, 57, 57, 58, 82, 68, 103),
    "AF":  (1.9, 4, 5, 7, 12, 12, 30),
    "WA":  (5.8, 11, 11, 12, 20, 15, 35),
    "SA":  (11.1, 18, 22, 25, 43, 45, 85),
    "CEA": (94.5, 136, 144, 215, 264, 242, 328),
    "SEA": (0, 0, 0, 3, 7, 4, 18),
    "OC":  (0, 0, 0, 0, 0, 0, 2),
}
THETA_KV = {"CA": 0.28, "SAM": 0.14, "WEU": 0.17794117647, "EEU": 0.1325, "AF": 0.16688725490,
            "WA": 0.13375, "SA": 0.185, "CEA": 0.135, "SEA": 0.16583333, "OC": 0.18}
REGIONS = list(THETA_KV.keys())

import pris_loader as pl

PRIS_UNITS = pl.load_units(str(NB_DIR / "rds2_2025_units.csv"))
_milestones_2024 = {r: REGION_MILESTONES[r][0] for r in REGIONS} | {"US": US_GW_2024}
_issues = pl.validate_units(PRIS_UNITS[PRIS_UNITS["status"] == "operational"], _milestones_2024)
assert not _issues, _issues
FLEET = PRIS_UNITS[PRIS_UNITS["status"] == "operational"]
UC_UNITS = PRIS_UNITS[PRIS_UNITS["status"] == "under construction"]

def retirement_flow_gw(key, life_yr):
    """GW retiring per model year, from each unit's real grid-connection date + a lifetime rule."""
    return pl.retirement_schedule(FLEET, pd.Index(YEARS), lifetime_years=life_yr,
                                  region=key).to_numpy()

PIPE_END = 2030   # pipeline's real visibility horizon; structurally zero after 2030 (mc S3)
PIPELINE_GW = {r: pl.committed_pipeline(UC_UNITS[UC_UNITS["region"] == r],
                                        pd.Index(np.arange(2025, PIPE_END + 1)),
                                        lead_time_months=72).to_dict()
               for r in REGIONS}
# Under-construction units are excluded: 5 carry PLANNED grid dates (not completed experience)
_ever = PRIS_UNITS[(PRIS_UNITS["status"] != "under construction")
                   & PRIS_UNITS["grid_connection"].notna()]
HIST_UNITS = {r: int((_ever["region"] == r).sum()) for r in REGIONS + ["US"]}
US_LICENSE_LIFE = 80.0
ONE_FACTOR_DELTA = 0.0

def net_path_gw(region, u):
    """Regional net capacity path: interpolate the 2024/2030/2040/2050 milestones, positioned
    between Low and High by the global deployment draw u."""
    c = REGION_MILESTONES[region]
    vals = [c[0], c[1] + u*(c[2]-c[1]), c[3] + u*(c[4]-c[3]), c[5] + u*(c[6]-c[5])]
    return np.interp(YEARS, [2024, 2030, 2040, 2050], vals)

def gross_additions_gw(region, u):
    """The retirement identity: builds = max(0, capacity change + retirements); the committed
    pipeline REPLACES the interpolated path through PIPE_END = 2030 (real units; structurally
    zero after 2030, so it only shapes the pre-anchor record - see mc_cost_trajectories S3)."""
    dnet = np.diff(net_path_gw(region, u), prepend=net_path_gw(region, u)[0])
    ret = retirement_flow_gw(region, 65.0 + 5.0*u)   # IAEA-calibrated lives, 65+5u (mc S3; 2026-08-05)
    g = np.clip(dnet + ret, 0.0, None)
    pipe_mask = YEARS <= PIPE_END
    g[pipe_mask] = [PIPELINE_GW[region].get(int(y), 0.0) for y in YEARS[pipe_mask]]
    return g

# Foreign stocks on the u grid (companion S3): unsplit, 1-GW basis - the engine's cross-firm
# channel does not distinguish designs (companion S5 Step 5). The IAEA SMR-share split of the
# foreign fleet lives in the main notebook's S14 own-routing sensitivity, not in any engine.
REGION_STOCK = {}
for r in REGIONS:
    m_a = np.empty((len(U_GRID), T))
    for gi, u in enumerate(U_GRID):
        flow = gross_additions_gw(r, u)
        flow = flow.copy(); flow[YEARS <= ANCHOR] = 0.0
        m_a[gi] = np.cumsum(flow) / UNIT_FOREIGN_GW
    REGION_STOCK[r] = m_a

# US gross GW added per year under each schedule (post-anchor), and the program unit
# counts under 100%-SMR (N_US_SMR) and 100%-large (N_US_LARGE, the large100 comparators)
# builds. ZEROS_T is the loser tech's (international-only) channel in either direction.
US_RET = retirement_flow_gw("US", US_LICENSE_LIFE)
GW_ADD, N_US_SMR, N_US_LARGE = {}, {}, {}
for name in SCHED_ORDER:
    cap = US_SCHEDULES[name] - OFFSET_GW
    dnet = np.diff(cap, prepend=cap[0])
    flow = np.clip(dnet + US_RET, 0.0, None)
    flow[YEARS <= ANCHOR] = 0.0
    GW_ADD[name] = flow
    N_US_SMR[name] = np.cumsum(flow) / TECH["smr"]["unit_gw"]
    N_US_LARGE[name] = np.cumsum(flow) / TECH["large"]["unit_gw"]
ZEROS_T = np.zeros(T)

print(f"PRIS fleet loaded: {len(FLEET)} operational, {len(UC_UNITS)} under construction")
print("SMR units by 2050 under 100%-SMR deployment, per schedule:")
for name in SCHED_ORDER:
    print(f"  {name:22s} {N_US_SMR[name][-1]:6.0f} SMR units ({GW_ADD[name].sum():.0f} GW)")

PRIS fleet loaded: 417 operational, 62 under construction
SMR units by 2050 under 100%-SMR deployment, per schedule:
  EIA 2026 AEO high          69 SMR units (21 GW)
  Abou-Jaoude mod           121 SMR units (36 GW)
  IAEA high                 252 SMR units (76 GW)
  McKinsey GEP 2025         345 SMR units (103 GW)
  COP28 tripling pledge     678 SMR units (203 GW)
  2025 EO                  1011 SMR units (303 GW)


In [4]:
# The OCC engine — verbatim companion port (mc_cost_trajectories.ipynb S5).
H_ALL_W = sum(THETA_KV[r]*HIST_UNITS[r] for r in REGIONS)

def lag1(a):
    """One-year completion lag along the time axis: the stock ENTERING year t is the builds
    completed through t-1. Cumulative arrays stay end-of-year for accounting; the lag is
    applied here, at the pricing boundary - without it the first post-anchor cohort would
    price on its own not-yet-built units (below BOAK). Companion-identical (mc S5)."""
    a = np.asarray(a, float)
    out = np.zeros_like(a)
    out[..., 1:] = a[..., :-1]
    return out

OTHER_TECH = {"large": "smr", "smr": "large"}
X_IN_COL = {"large": "x_sl", "smr": "x_ls"}     # incoming cross-tech fraction, per receiving tech

def experience_channels(world, tech, n_us, n_oth=None):
    """The two experience stocks O_d(t) (own) and A_d(t) (cross-firm), each (n_draws, T).
    Companion-identical (mc S5): international experience enters the CROSS-FIRM channel at
    weight s*theta, unsplit across technologies (the K&V flows are normalized to the
    domestic inter-firm citation baseline - omega prices the firm wall, theta the border);
    the drawn incoming x scales the other tech's US program (n_oth) into the same channel."""
    n = len(world)
    u_r = np.repeat(world["u"].values[:, None], len(REGIONS), axis=1)
    if ONE_FACTOR_DELTA > 0:
        u_r = np.clip(u_r + rng_stream("one_factor_noise").uniform(
            -ONE_FACTOR_DELTA, ONE_FACTOR_DELTA, (n, len(REGIONS))), 0, 1)
    gi = np.clip(np.rint(u_r * (len(U_GRID)-1)).astype(int), 0, len(U_GRID)-1)
    S_kv = np.zeros((n, T))
    for j, r in enumerate(REGIONS):
        S_kv += THETA_KV[r] * REGION_STOCK[r][gi[:, j], :]
    S_kv = lag1(S_kv)                                    # stock entering year t (built thru t-1)
    N_us = lag1(np.broadcast_to(np.asarray(n_us, float), (n, T)))
    conv = world["conv_full"].values[:, None].astype(float)
    m = world["n_vendors"].values[:, None].astype(float)
    s = world["s"].values[:, None]
    hist_us = HIST_UNITS["US"] if tech == "large" else 0.0    # D8': US LWR history -> large only
    own0 = N_BOAK_UNITS + conv*hist_us/m
    own = own0 + N_us/m
    oth = (m-1.0)*own0 + N_us*(m-1.0)/m \
          + s*(conv*H_ALL_W + S_kv)
    if n_oth is not None:                                # cross-tech: x * the other tech's program
        x = world[X_IN_COL[tech]].values[:, None]
        oth = oth + x*lag1(np.broadcast_to(np.asarray(n_oth, float), (n, T)))
    return own, oth

CES_EPS = 1e-8

def occ_paths_ces(world, tech, n_us, n_oth=None, ces_rho=None):
    """Production OCC engine (companion D11/D12, verbatim). Returns $/kW (2022 USD)."""
    own, oth = experience_channels(world, tech, n_us, n_oth=n_oth)
    assert own.min() >= N_BOAK_UNITS and oth.min() >= N_BOAK_UNITS, "stock below anchor base; log unsafe"
    lr = world[f"lr_{tech}"].values[:, None]
    b1, b2 = np.log2(1.0 - lr), np.log2(1.0 - OMEGA*lr)
    b, w = -(b1 + b2), b1/(b1 + b2)
    rho = world["ces_rho"].values if ces_rho is None else np.full(len(world), float(ces_rho))
    rho = rho[:, None]
    geo = np.abs(rho) < CES_EPS
    lnO, lnA = np.log(own), np.log(oth)
    with np.errstate(over="raise"):
        lnE = np.where(geo, w*lnO + (1.0-w)*lnA,
                       lnO + np.log1p((1.0-w)*np.expm1(np.where(geo, 0.0, rho)*(lnA - lnO)))
                           / np.where(geo, 1.0, rho))
    return world[f"boak_{tech}"].values[:, None] * np.exp(-b*(lnE - lnE[:, [yi(ANCHOR)]]))

def percentile_table(paths, ps=(5, 25, 50, 75, 95)):
    """Yearly percentiles of a (n_draws, T) trajectory array."""
    return pd.DataFrame({f"P{p}": np.percentile(paths, p, axis=0) for p in ps}, index=YEARS)

In [5]:
# Durations + ReEDS financing — verbatim companion ports (S6-S7 there).
DUR_UNITS = pl.duration_panel(PRIS_UNITS)
DUR_UNITS = DUR_UNITS[DUR_UNITS["family"].isin(["AP1000", "APR1400", "HPR1000"])].reset_index(drop=True)
DUR_UNITS = DUR_UNITS.rename(columns={"reactor_name": "unit"})

def fit_family_fe(df):
    """ln(duration) = family intercepts + b*ln(family sequence); returns (coefficients, residual sd)."""
    X = np.column_stack([np.ones(len(df)),
                         (df["family"] == "APR1400").astype(float),
                         (df["family"] == "HPR1000").astype(float),
                         np.log(df["family_seq"].astype(float))])
    Y = np.log(df["months"].values)
    beta, *_ = np.linalg.lstsq(X, Y, rcond=None)
    resid = Y - X @ beta
    return beta, resid.std(ddof=min(4, len(df)-1))

beta_hat, sig_hat = fit_family_fe(DUR_UNITS)

INL_DUR_MOD = np.array([118.0, 88.0, 74.0, 67.0, 62.0, 59.0, 56.0, 54.0, 52.0, 50.0])
INL_DUR_OPT = np.array([118.0, 70.0, 57.0, 49.0, 45.0, 42.0, 40.0, 38.0, 37.0, 36.0])
SMR_DUR_RATIO = 55.0/82.0
SMR_DUR_FLOOR = 43.0

def duration_paths(world, tech, n_us):
    """Construction duration (months, FNC -> COD) per draw-year, from the INL series curves."""
    n = len(world)
    N_us = lag1(np.broadcast_to(np.asarray(n_us, float), (n, T)))   # completed units only
    n_own = N_us / world["n_vendors"].values[:, None]
    series = np.clip(2 + np.floor(n_own/2.0).astype(int), 2, len(INL_DUR_MOD))
    opt, mod = np.take(INL_DUR_OPT, series-1), np.take(INL_DUR_MOD, series-1)
    base = opt + world["dur_lambda"].values[:, None]*(mod - opt)
    if tech == "smr":
        base = np.maximum(base * SMR_DUR_RATIO, SMR_DUR_FLOOR)
    return base * np.exp(world["dur_z"].values[:, None]*sig_hat)

FIN_DIR = REPO_ROOT / "inputs" / "financials"

def load_reeds_financials():
    """ReEDS financial inputs on the YEARS grid, exactly as reeds/financials.py derives them."""
    sys_fin = pd.read_csv(FIN_DIR / "financials_sys_ATB2024.csv")
    infl = pd.read_csv(FIN_DIR / "inflation_default.csv")
    sys_fin = sys_fin.merge(infl, on="t", how="left")
    sys_fin["d_nom"] = ((1 - sys_fin["debt_fraction"]) * (sys_fin["rroe_nom"] - 1)
                        + sys_fin["debt_fraction"] * (sys_fin["interest_rate_nom"] - 1)
                          * (1 - sys_fin["tax_rate"]) + 1)
    sys_fin["d_real"] = sys_fin["d_nom"] / sys_fin["inflation_rate"]
    tech_fin = pd.read_csv(FIN_DIR / "financials_tech_ATB2024.csv")
    tech_fin = tech_fin[tech_fin["i"].isin(["Nuclear", "Nuclear-SMR"])]
    dep = pd.read_csv(FIN_DIR / "depreciation_schedules_default.csv")
    cs = pd.read_csv(FIN_DIR / "construction_schedules_default.csv")

    def on_years(col):
        s = sys_fin.set_index("t")[col].reindex(range(1990, YEARS[-1] + 1)).ffill()
        return s.loc[YEARS].to_numpy(float)

    out = {"interest_base": on_years("interest_rate_nom"), "tax_rate": on_years("tax_rate"),
           "d_nom": on_years("d_nom"), "d_real": on_years("d_real"),
           "pv_dep": {}, "risk_mult": {}, "eval_adj": {}, "sched": {}}
    SYS_EVAL_YEARS = 30
    sys_pvf_sum = (1 - (1/out["d_real"])**(SYS_EVAL_YEARS - 1)) / (out["d_real"] - 1.0) + 1
    REEDS_TECH_NAME = {"large": "Nuclear", "smr": "Nuclear-SMR"}
    for tech, iname in REEDS_TECH_NAME.items():
        row = tech_fin[tech_fin["i"] == iname].iloc[-1]
        dep_frac = dep[str(int(row["depreciation_sch"]))].to_numpy(float)
        out["pv_dep"][tech] = np.array([np.sum(dep_frac / dn**np.arange(1, 22))
                                        for dn in out["d_nom"]])
        eval_p = float(row["eval_period"])
        out["risk_mult"][tech] = 1.0 + float(row["finance_diff_real"]) * (
            (1 - (1/out["d_real"])**eval_p) / (out["d_real"] - 1.0))
        tech_pvf_sum = (1 - (1/out["d_real"])**(eval_p - 1)) / (out["d_real"] - 1.0) + 1
        out["eval_adj"][tech] = sys_pvf_sum / tech_pvf_sum
        frac = pd.to_numeric(cs[str(row["construction_sch"])], errors="coerce").fillna(0.0).to_numpy()
        nz = np.flatnonzero(frac > 0)
        assert nz.size and nz[-1] - nz[0] + 1 == nz.size, \
            f"{iname}: spend profile has interior zeros; frac[frac>0] would shift spend years"
        out["sched"][tech] = frac[nz[0]:nz[-1] + 1]
    return out

FIN = load_reeds_financials()

def _resample_schedule(frac, n_years):
    """Stretch/compress a spend-fraction profile to `n_years` bins (sum to 1)."""
    frac = np.asarray(frac, float)
    n0 = len(frac)
    if n_years == n0:
        return frac / frac.sum()
    cdf = np.concatenate([[0.0], np.cumsum(frac)])
    xq = np.linspace(0.0, 1.0, n_years + 1)
    cdf_q = np.interp(xq, np.linspace(0.0, 1.0, n0 + 1), cdf)
    new = np.diff(cdf_q)
    return new / new.sum()

def ccmult_from_duration(duration_mo, interest_base, canonical_frac):
    """Construction-financing (IDC) multiplier for a learned duration (verbatim ReEDS port)."""
    n_years = int(round(duration_mo / 12.0))
    n_years = max(1, min(n_years, 10))
    x = _resample_schedule(canonical_frac, n_years)
    exps = np.arange(n_years) + 0.5
    return 1.0 + float(np.sum(x * (interest_base ** exps - 1.0)))

def ccmult_grid(dur_months, tech):
    """Vectorized ccmult for a (n_draws, T) duration array."""
    ib = FIN["interest_base"]
    table = np.empty((10 + 1, T))
    for n in range(1, 11):
        x = _resample_schedule(FIN["sched"][tech], n)
        exps = np.arange(n) + 0.5
        table[n] = 1.0 + (x[:, None] * (ib[None, :] ** exps[:, None] - 1.0)).sum(axis=0)
    n_years = np.clip(np.round(np.asarray(dur_months)/12.0).astype(int), 1, 10)
    return table[n_years, np.arange(T)[None, :]]

def fin_mult_rest(tech, use_itc=False):
    """Everything in ReEDS's cost_cap_fin_mult EXCEPT ccmult, per year (T,) — verbatim port."""
    assert not use_itc, "ITC variant deliberately not implemented"
    tax, pv_dep = FIN["tax_rate"], FIN["pv_dep"][tech]
    return (1.0/(1.0 - tax)) * (1.0 - tax*pv_dep) * FIN["risk_mult"][tech] * FIN["eval_adj"][tech]

FIN_REST = {tech: fin_mult_rest(tech) for tech in TECH}
DISC = np.cumprod(np.where(YEARS > ANCHOR, FIN["d_real"], 1.0))   # discount to 2030
print(f"foundations loaded: duration panel {len(DUR_UNITS)} units (sd {sig_hat:.3f}); "
      f"ccmult(72mo, 8%, '6') = {ccmult_from_duration(72.0, 1.08, FIN['sched']['large']):.6f}")

foundations loaded: duration panel 21 units (sd 0.116); ccmult(72mo, 8%, '6') = 1.268124


In [6]:
# Ported verbatim from mc_cost_trajectories.ipynb S8.
RHO_SETS = {"zero": 0.0, "moderate": -0.3, "strong": -0.6}
LATENTS = ["lr", "boak", "u"]

def corr_matrix(rho_key):
    C = np.eye(3)
    C[0, 1] = C[1, 0] = RHO_SETS[rho_key]
    ev = np.linalg.eigvalsh(C)
    assert ev.min() > 1e-10, f"correlation matrix '{rho_key}' is not positive definite: {ev}"
    return C

def draw_world(n, rho_key, rng):
    """Draw n worlds: every uncertain input, one row per draw (companion S8, verbatim)."""
    Z = rng.standard_normal((n, 3)) @ np.linalg.cholesky(corr_matrix(rho_key)).T
    U = stats.norm.cdf(Z)                                   # Gaussian copula -> uniforms
    w = pd.DataFrame(index=range(n))
    for tech in TECH:
        t = TECH[tech]
        w[f"lr_{tech}"] = t["lr_lo"] + U[:, 0]*(t["lr_hi"] - t["lr_lo"])      # comonotone across techs
        w[f"boak_{tech}"] = t["boak_lo"] + U[:, 1]*(t["boak_hi"] - t["boak_lo"])
    w["u"] = U[:, 2]                                        # global deployment position
    w["s"] = rng.uniform(0.0, 1.0, n)                       # spillover scale
    w["n_vendors"] = rng.integers(4, 9, n)                  # vendor count m
    w["conv_full"] = rng.integers(0, 2, n)                  # 0 = tiny-base, 1 = full-stock
    w["ces_rho"] = rng.choice(CES_RHO_GRID, size=n)         # channel-substitution stratum
    w["dur_lambda"] = rng.uniform(0.0, 1.0, n)              # duration scenario position
    w["dur_z"] = rng.standard_normal(n)                     # persistent duration project noise
    w["x_ls"] = rng.uniform(0.0, X_SPILL_MAX, n)            # cross-tech spillover: large -> SMR
    w["x_sl"] = rng.uniform(0.0, X_SPILL_MAX, n)            # cross-tech spillover: SMR -> large
    return w

WORLDS, results = {}, {}
for sched in SCHED_ORDER:
    # The COMPANION's stream names, deliberately: identical worlds to mc_cost_trajectories.ipynb
    world = draw_world(N_DRAWS, COPULA_SET, rng_stream(f"world/{SCEN_TOKEN[sched]}"))
    WORLDS[sched] = world
    r = {}
    for tech, n_us, n_oth in (("smr", N_US_SMR[sched], None),
                              ("large", ZEROS_T, N_US_SMR[sched])):   # loser: x * SMR program
        r[f"occ_{tech}"] = occ_paths_ces(world, tech, n_us, n_oth=n_oth)
        r[f"dur_{tech}"] = duration_paths(world, tech, n_us)
        r[f"ccmult_{tech}"] = ccmult_grid(r[f"dur_{tech}"], tech)
        r[f"fincapex_{tech}"] = r[f"occ_{tech}"] * r[f"ccmult_{tech}"] * FIN_REST[tech][None, :]
    results[sched] = r

print(f"drew {N_DRAWS} worlds x {len(SCHED_ORDER)} schedules (companion draw, comonotone)")
print("\n2050 SMR OCC and financed CAPEX by schedule ($/kW 2022, P5 / P50 / P95):")
for sched in SCHED_ORDER:
    o = results[sched]["occ_smr"][:, yi(2050)]
    f = results[sched]["fincapex_smr"][:, yi(2050)]
    print(f"  {sched:22s} OCC {np.percentile(o,5):5,.0f} /{np.percentile(o,50):6,.0f} /"
          f"{np.percentile(o,95):6,.0f}   financed {np.percentile(f,5):5,.0f} /"
          f"{np.percentile(f,50):6,.0f} /{np.percentile(f,95):6,.0f}")

drew 10000 worlds x 6 schedules (companion draw, comonotone)

2050 SMR OCC and financed CAPEX by schedule ($/kW 2022, P5 / P50 / P95):
  EIA 2026 AEO high      OCC 3,480 / 5,494 / 8,151   financed 4,597 / 7,280 /10,831
  Abou-Jaoude mod        OCC 3,101 / 5,110 / 7,897   financed 4,114 / 6,779 /10,432
  IAEA high              OCC 2,506 / 4,493 / 7,441   financed 3,307 / 5,967 / 9,878
  McKinsey GEP 2025      OCC 2,304 / 4,291 / 7,277   financed 3,067 / 5,705 / 9,681
  COP28 tripling pledge  OCC 1,891 / 3,779 / 6,890   financed 2,505 / 5,016 / 9,134
  2025 EO                OCC 1,693 / 3,483 / 6,610   financed 2,251 / 4,615 / 8,776


In [7]:
# --- Non-CAPEX costs for the NPV ranking (adopted 2026-08-03) ---
# All validation lives in npv_winner_check.ipynb: the winner survives the full NPV in >99%
# of draws, but WHICH draw sits nearest each P5/P50/P95 target moves -- so the selection
# here carries the full NPV. Convention: FOM and VOM sit at the same percentile of their
# ranges (ATB 2024 advanced -> conservative) as the draw's 2030 anchor cost does in its
# range -- comonotone with the cost dial, never independent sensitivities. CF equals ReEDS's
# own availability (avail, replicated bit-exactly from raw inputs in npv_winner_check N1);
# fuel is heat rate x AEO 2026 uranium; annual costs are levelized with ReEDS's
# pvf_onm = 1/crf (30-yr window at the system real rate).
PLANTCHAR_DIR = REPO_ROOT / "inputs" / "plant_characteristics"
EVAL_YEARS = 30
_d_real = float(DISC[-1] / DISC[-2])
_ks = np.arange(1, EVAL_YEARS + 1)
PVF_ONM = float((_d_real ** -_ks).sum())
assert round(1.0 / PVF_ONM, 5) == 0.06773   # ReEDS crf.csv pin (validated in npv_winner_check)

# CF: the avail number computed and QA'd by npv_winner_check.ipynb (national mean; identical
# for both techs by ReEDS's prime-mover construction)
_cf = pd.read_csv(NB_DIR / "exports" / "npv_winner_summary.csv")["CF"].unique()
assert len(_cf) == 1 and 0.89 < float(_cf[0]) < 0.92
CF_AVAIL = float(_cf[0])

_ATB_STEM = {"large": "nuclear", "smr": "nuclear-smr"}
OM_RANGE = {}
for tech in TECH:
    ends = {s: pd.read_csv(PLANTCHAR_DIR / f"{_ATB_STEM[tech]}_ATB_2024_{s}.csv")
                 .set_index("t").loc[2030]
            for s in ("advanced", "conservative")}
    # the ATB scenario axis IS the MC's anchor-cost axis -- one percentile indexes both
    assert (float(ends["advanced"]["capcost"]), float(ends["conservative"]["capcost"])) \
        == (TECH[tech]["boak_lo"], TECH[tech]["boak_hi"]), tech
    assert float(ends["advanced"]["heatrate"]) == float(ends["conservative"]["heatrate"])
    OM_RANGE[tech] = {
        "fom": (float(ends["advanced"]["fom"]), float(ends["conservative"]["fom"])),
        "vom": (float(ends["advanced"]["vom"]), float(ends["conservative"]["vom"])),
        "hr": float(ends["advanced"]["heatrate"]),
    }

def u1_of(world):
    """The draw's cost-dial percentile (shared by both techs -- comonotone)."""
    t = TECH["large"]
    return ((world["boak_large"] - t["boak_lo"]) / (t["boak_hi"] - t["boak_lo"])).to_numpy()

def om_at(tech, U1):
    """FOM ($/kW-yr) and VOM ($/MWh) at cost-dial percentile U1 (advanced -> conservative)."""
    lo_f, hi_f = OM_RANGE[tech]["fom"]
    lo_v, hi_v = OM_RANGE[tech]["vom"]
    return lo_f + U1 * (hi_f - lo_f), lo_v + U1 * (hi_v - lo_v)

# uranium AEO 2026 baseline: 2025$ -> 2022$, flat after 2050; per-build-year 30-yr fuel PV
_defl = pd.read_csv(REPO_ROOT / "inputs" / "financials" / "deflator.csv")
_defl.columns = ["year", "deflator"]
_defl = _defl.set_index("year")["deflator"]
_Pu = (pd.read_csv(REPO_ROOT / "inputs" / "fuelprices" / "uranium_AEO_2026_baseline.csv")
       .set_index("year")["cost"] * float(_defl.loc[2025] / _defl.loc[2022]))
_Pu = _Pu.reindex(range(int(YEARS.min()), int(YEARS.max()) + EVAL_YEARS + 1)).ffill()
MWH_PER_KWYR = 8.760 * CF_AVAIL                    # MWh generated per kW-yr at CF = avail
_FUEL_PV = {tech: MWH_PER_KWYR * OM_RANGE[tech]["hr"] * np.array(
    [(_Pu.loc[t + 1: t + EVAL_YEARS].to_numpy() * (_d_real ** -_ks)).sum() for t in YEARS])
    for tech in TECH}

def om_npv_per_kw(tech, U1):
    """(n, T) 30-yr PV per kW of capacity built in each year: FOM + VOM + fuel, 2022$."""
    fom, vom = om_at(tech, np.asarray(U1)[:, None])
    return PVF_ONM * (fom + MWH_PER_KWYR * vom) + _FUEL_PV[tech][None, :]

# the convention's check values (SMR at U1 = 0.75): boak $8,875/kW -> FOM $191.5, VOM $2.65
_f, _v = om_at("smr", 0.75)
assert (round(float(_f), 1), round(float(_v), 2)) == (191.5, 2.65)
print(f"NPV-ranking inputs: CF = avail = {CF_AVAIL:.4f} (npv_winner_check), "
      f"pvf_onm = 1/crf = {PVF_ONM:.4f};")
print("  FOM/VOM comonotone with the draw's cost percentile across ATB advanced->conservative;")
print("  30-yr PV of non-CAPEX per kW built 2031, at U1=0.5 ($/kW 2022): "
      f"large {om_npv_per_kw('large', np.array([0.5]))[0, yi(2031)]:,.0f}, "
      f"smr {om_npv_per_kw('smr', np.array([0.5]))[0, yi(2031)]:,.0f}")


NPV-ranking inputs: CF = avail = 0.9044 (npv_winner_check), pvf_onm = 1/crf = 14.7637;
  FOM/VOM comonotone with the draw's cost percentile across ATB advanced->conservative;
  30-yr PV of non-CAPEX per kW built 2031, at U1=0.5 ($/kW 2022): large 3,303, smr 3,245


In [8]:
SHORT = {"eia_aeo_high": "eia", "abou_jaoude": "aj", "iaea_high": "iaea",
         "mckinsey": "mck", "cop28": "cop28", "eo2025": "eo"}

def rank_smr(sched):
    """Program-NPV score per draw: discounted GW-weighted SMR financed CAPEX plus the 30-yr
    PV of FOM/VOM/fuel at the draw's own O&M percentile (lower = cheaper)."""
    w_t = GW_ADD[sched] / DISC
    npv = results[sched]["fincapex_smr"] + om_npv_per_kw("smr", u1_of(WORLDS[sched]))
    return npv @ w_t

print("cost object defined: rank_smr(sched) — the program-NPV scale every diagnostic, "
      "bound, and placement below is measured on")

cost object defined: rank_smr(sched) — the program-NPV scale every diagnostic, bound, and placement below is measured on


In [9]:
# Program-NPV score vectors (labels the selected worlds' pctile_in_MC; the selection
# itself comes from u80, never from these scores).
mc_score = {s: rank_smr(s) for s in SCHED_ORDER}
print("program-NPV scores computed for all six schedules")


program-NPV scores computed for all six schedules


In [10]:
# --- Production-case selection (v10, 2026-08-06): the P5/P50/P95 joint draws per schedule ---
# Rank the 10k drawn worlds by the program NPV (mc_score) and take the draws at the
# P5/P50/P95 ranks (rank = ceil(q*(N-1)) of the stable argsort). The cases are actual joint
# draws -- plausible worlds with probability mass -- selected as a constrained optimizer over
# the drawn (plausible) set; the lo/hi bounds above remain the appendix possibility frontier
# over the priors' full support.
PCT_TAGS = {"p05": 0.05, "p50": 0.50, "p95": 0.95}
SELECTED = {}                # SELECTED[(sched, tag)] -> {"idx": draw index, "score": float}
_sel_rows = []
for sched in SCHED_ORDER:
    order = np.argsort(mc_score[sched], kind="stable")
    for tag, q in PCT_TAGS.items():
        idx = int(order[int(np.ceil(q * (N_DRAWS - 1)))])
        SELECTED[(sched, tag)] = {"idx": idx, "score": float(mc_score[sched][idx])}
        _sel_rows.append({"schedule": sched, "percentile": tag, "draw_index": idx,
                          "score": float(mc_score[sched][idx]),
                          "pctile_in_MC": round(100*float(
                              (mc_score[sched] < mc_score[sched][idx]).mean()), 3),
                          **{c: float(WORLDS[sched].iloc[idx][c])
                             for c in WORLDS[sched].columns}})
sel_df = pd.DataFrame(_sel_rows).set_index(["schedule", "percentile"])
sel_df.to_csv(EXPORTS / "selected_draws.csv")

# Draw-identity assert: each selected draw's parameter row must match the companion's
# per-draw export (extends QA-0's sample-space identity to the cases themselves).
_npz_sel_path = NB_DIR / "exports" / "mc_perdraw.npz"
_sel_checked = 0
if _npz_sel_path.exists():
    _npz_sel = np.load(_npz_sel_path, allow_pickle=False)
    _sel_cols = list(_npz_sel["world_columns"].astype(str))
    for (sched, tag), s in SELECTED.items():
        tok = SCEN_TOKEN[sched]
        if _npz_sel[f"worlds_{tok}"].shape[0] != N_DRAWS:
            print("selected-draw identity SKIPPED: mc_perdraw.npz draw count differs "
                  "from this run -- re-run the companion notebook to compare")
            break
        assert _sel_cols == list(WORLDS[sched].columns), "world column layout diverged"
        assert np.allclose(_npz_sel[f"worlds_{tok}"][s["idx"]],
                           WORLDS[sched].iloc[s["idx"]].to_numpy(dtype=float),
                           rtol=1e-9, atol=0), (sched, tag)
        _sel_checked += 1
    if _sel_checked:
        print(f"selected-draw identity vs mc_perdraw.npz: all {_sel_checked} case rows match "
              "their registered draw (rtol 1e-9)")
else:
    print("mc_perdraw.npz not found -- selected-draw identity check skipped "
          "(run the companion notebook to enable it)")

print("\nSelected production draws (registered to exports/smr100/selected_draws.csv):")
print(sel_df[["draw_index", "score", "pctile_in_MC"]].to_string())

selected-draw identity vs mc_perdraw.npz: all 18 case rows match their registered draw (rtol 1e-9)

Selected production draws (registered to exports/smr100/selected_draws.csv):
                                  draw_index         score  pctile_in_MC
schedule              percentile                                        
EIA 2026 AEO high     p05               2329  1.074627e+05           5.0
                      p50               9573  1.481451e+05          50.0
                      p95               6874  1.950652e+05          95.0
Abou-Jaoude mod       p05               6077  1.399636e+05           5.0
                      p50                242  1.971983e+05          50.0
                      p95                162  2.659700e+05          95.0
IAEA high             p05               8423  3.268254e+05           5.0
                      p50               4993  4.708006e+05          50.0
                      p95               7602  6.512874e+05          95.0
McKinsey GEP 2025   

In [11]:
# Continuity pin: the re-derived smr100 selection == the frozen Step-1 registration
# (the horizon runs rerun exactly those p50 cases, so this pin licenses the reuse).
_frozen_sel_full = pd.read_csv(NB_DIR / "exports" / "smr100" / "selected_draws.csv",
                               index_col=["schedule", "percentile"])
assert (_frozen_sel_full["draw_index"] == sel_df["draw_index"]).all(), "selection diverged"
print("continuity pin PASSED: smr100 selections identical to the frozen registration")


continuity pin PASSED: smr100 selections identical to the frozen registration


## The frozen batch spec

`u80_batch_spec.csv` is the single source of truth for the ITC-arm and horizon run names,
blocks, schedules, and draw indices (v2.2: 54 rows, no reserve). The horizon rows must sit
on the frozen smr100 p50 draws — asserted against `exports/smr100/selected_draws.csv` — and
the live runs must have distinct worlds within each schedule. The counts below are DERIVED
from u80, never hard-coded, and echoed against the v2.2 registration (48 / 6).


In [12]:
u80 = pd.read_csv(RD_EXPORTS / "u80_batch_spec.csv")
u81 = pd.read_csv(RD_EXPORTS / "u81_run_schedules.csv")
assert (u80["block"] == "reserve").sum() == 0, "v2.2 retired the reserve"

STEM2SCHED = {SHORT[SCEN_TOKEN[s]]: s for s in SCHED_ORDER}   # 'aj' -> 'Abou-Jaoude mod'
LIVE_BLOCKS = ["envelope", "boundary", "hybrid"]
live = u80[u80["block"].isin(LIVE_BLOCKS)].reset_index(drop=True)
horiz = u80[u80["block"] == "horizon"].reset_index(drop=True)
N_LIVE, N_HZ = len(live), len(horiz)
assert (N_LIVE, N_HZ) == (48, 6), (N_LIVE, N_HZ)          # methods.md v2.2 registration
assert len(u80) == N_LIVE + N_HZ
assert (live["max_rate_on_world"] < 1.0).all()
assert (live["max_rate_on_world"] <= 0.95).all()          # the registered rate screen
# u80's own row order is envelope -> boundary -> hybrid -> horizon; the casefile keeps it
# (anchor block inserted before horizon; horizon last by construction). Assert:
blocks_seq = list(u80["block"])
_order = ["envelope", "boundary", "hybrid", "horizon"]
assert blocks_seq == sorted(blocks_seq, key=_order.index), "u80 block order changed"
BLOCK_N = {b: int((u80["block"] == b).sum()) for b in _order}

# one distinct world per live run within a schedule; case token rd_{stem}_d{draw}
live_worlds = list(zip(live["schedule"], live["draw_index"].astype(int)))
assert len(set(live_worlds)) == N_LIVE, "live worlds are not all distinct"
live = live.assign(case=[f"rd_{ab}_d{w}" for ab, w in live_worlds])

# horizon rows sit on the frozen p50 draws
_frozen_sel = pd.read_csv(NB_DIR / "exports" / "smr100" / "selected_draws.csv")
_p50 = {r["schedule"]: int(r["draw_index"])
        for _, r in _frozen_sel[_frozen_sel["percentile"] == "p50"].iterrows()}
for _, r in horiz.iterrows():
    assert int(r["draw_index"]) == _p50[STEM2SCHED[r["schedule"]]], r["run"]

# u81 covers exactly the live runs, and every offered rate is < 1
assert set(u81["run"]) == set(live["run"])
assert (u81["rate_on_world"] < 1.0).all() and (u81["rate_on_world"] > 0).all()
print(f"batch spec loaded: {N_LIVE} live runs ({N_LIVE} distinct worlds; blocks {BLOCK_N}), "
      f"{N_HZ} horizon reruns on the frozen p50 draws, no reserve (v2.2)")


batch spec loaded: 48 live runs (48 distinct worlds; blocks {'envelope': 27, 'boundary': 7, 'hybrid': 14, 'horizon': 6}), 6 horizon reruns on the frozen p50 draws, no reserve (v2.2)


## Anchor densification (v2.2): the p25/p75 mandate-arm worlds

Registered in `rate_design/methods.md` v2.2 (2026-09-04, before any run). Selection = the
frozen smr100 rule at two new quantiles: `idx = argsort(program-NPV score, stable)[ceil(q ·
(N − 1))]`, q = 0.25 / 0.75 — the same `rank_smr` ranking and the same index arithmetic
that placed p05/p50/p95 (continuity-pinned above). The frozen `selected_draws.csv` is never
rewritten; the new picks go to `u82_anchor_spec.csv` (rate_design exports) and
`exports/smr100/selected_draws_p25p75.csv`. Asserted distinct from the 18 frozen anchors
and from every ITC-arm world. Gates GA1/GA2 (methods.md v2.2) are adjudicated post-return.


In [13]:
ANCHOR_Q = {"p25": 0.25, "p75": 0.75}
_frozen18 = {(r["schedule"], int(r["draw_index"])) for _, r in _frozen_sel.iterrows()}
_live_set = {(STEM2SCHED[ab], int(w)) for ab, w in live_worlds}
anchor_rows, anchor_reg = [], []
for sched in SCHED_ORDER:
    order = np.argsort(mc_score[sched], kind="stable")
    stem = SHORT[SCEN_TOKEN[sched]]
    for tag, q in ANCHOR_Q.items():
        idx = int(order[int(np.ceil(q * (N_DRAWS - 1)))])
        assert (sched, idx) not in _frozen18, (sched, tag, idx, "collides with a frozen anchor")
        assert (sched, idx) not in _live_set, (sched, tag, idx, "collides with an ITC-arm world")
        score = float(mc_score[sched][idx])
        pct = round(100 * float((mc_score[sched] < score).mean()), 3)
        params = {c: float(WORLDS[sched].iloc[idx][c]) for c in WORLDS[sched].columns}
        anchor_rows.append({"run": f"smr100_{stem}_{tag}", "block": "anchor",
                            "schedule": stem, "percentile": tag, "q": q, "draw_index": idx,
                            "score": score, "pctile_in_MC": pct,
                            "expected": "diagnostic", "gate": "GA1/GA2", **params})
        anchor_reg.append({"schedule": sched, "percentile": tag, "draw_index": idx,
                           "score": score, "pctile_in_MC": pct, **params})
u82 = pd.DataFrame(anchor_rows)
N_ANCHOR = len(u82)
assert N_ANCHOR == 12 and u82["run"].is_unique
assert (abs(u82["pctile_in_MC"] - 100 * u82["q"]) <= 0.5).all(), "quantile placement drifted"
u82.to_csv(RD_EXPORTS / "u82_anchor_spec.csv", index=False)
pd.DataFrame(anchor_reg).set_index(["schedule", "percentile"]).to_csv(
    NB_DIR / "exports" / "smr100" / "selected_draws_p25p75.csv")
print(f"anchor densification: {N_ANCHOR} p25/p75 worlds selected by the frozen rule "
      f"(distinct from the 18 frozen anchors and the {N_LIVE} ITC-arm worlds)")
print(u82[["run", "schedule", "percentile", "draw_index", "pctile_in_MC"]].to_string(index=False))


anchor densification: 12 p25/p75 worlds selected by the frozen rule (distinct from the 18 frozen anchors and the 48 ITC-arm worlds)
             run schedule percentile  draw_index  pctile_in_MC
  smr100_eia_p25      eia        p25        7498          25.0
  smr100_eia_p75      eia        p75        6540          75.0
   smr100_aj_p25       aj        p25        3569          25.0
   smr100_aj_p75       aj        p75        9504          75.0
 smr100_iaea_p25     iaea        p25         516          25.0
 smr100_iaea_p75     iaea        p75        7323          75.0
  smr100_mck_p25      mck        p25        3229          25.0
  smr100_mck_p75      mck        p75        2237          75.0
smr100_cop28_p25    cop28        p25        8895          25.0
smr100_cop28_p75    cop28        p75        1940          75.0
   smr100_eo_p25       eo        p25        5003          25.0
   smr100_eo_p75       eo        p75        2664          75.0


In [14]:
# --- The 18 exported cases, built from the selected P5/P50/P95 joint draws ---
N_US_PROG = {"smr": N_US_SMR, "large": N_US_LARGE}   # program-tech unit counts per schedule

def build_case_rec(case, sched, tag, w1, score, extra, program="smr", score_dist=None):
    """One export-ready case record: parameters, U1-comonotone O&M, and the (T,) occ/dur
    path arrays -- shared by the production cases, the pilot bounds pair (no ATB-moderate
    special case anymore, v10), and the large100 comparators. `program` is the tech that
    receives the entire build program (the 'winner' field records it; the other tech rides
    the loser channel); `score_dist` is the ranking the case's percentile is measured in
    (defaults to the SMR program ranking mc_score)."""
    fomvom = {t: tuple(float(x) for x in om_at(t, float(u1_of(w1)[0]))) for t in TECH}
    r = w1.iloc[0]
    dist = mc_score[sched] if score_dist is None else score_dist
    rec = {"case": case, "schedule": sched, "scen_token": SCEN_TOKEN[sched], "case_type": tag,
           "winner": program,          # the tech assumed to receive the entire program
           "lr_large": float(r["lr_large"]), "lr_smr": float(r["lr_smr"]),
           "boak_large": float(r["boak_large"]), "boak_smr": float(r["boak_smr"]),
           "u": float(r["u"]), "s_spill": float(r["s"]),
           "x_ls": float(r["x_ls"]), "x_sl": float(r["x_sl"]),
           "n_vendors": int(r["n_vendors"]),
           "convention": "full-stock" if r["conv_full"] else "tiny-base",
           "ces_rho": float(r["ces_rho"]), "dur_lambda": float(r["dur_lambda"]),
           "dur_z": float(r["dur_z"]),
           "score": float(score),
           "pctile_in_MC": round(100*float((dist < score).mean()), 2),
           **extra,
           **{f"{k}_{t}": v for t in TECH for k, v in zip(("fom", "vom"), fomvom[t])}}
    other = "large" if program == "smr" else "smr"
    for tech, n_us, n_oth in ((program, N_US_PROG[program][sched], None),
                              (other, ZEROS_T, N_US_PROG[program][sched])):
        occ = occ_paths_ces(w1, tech, n_us, n_oth=n_oth)[0]
        dur = duration_paths(w1, tech, n_us)[0]
        rec[f"occ_{tech}"], rec[f"dur_{tech}"] = occ, dur    # (T,) arrays for the exports
        for y in (2030, 2040, 2050):
            rec[f"occ_{tech}_{y}"] = round(float(occ[yi(y)]), 0)
            rec[f"dur_{tech}_{y}"] = round(float(dur[yi(y)]), 0)
    return rec

CASES = {}           # CASES[case_name] -> the selected joint draw's record (+ (T,) arrays)
for sched in SCHED_ORDER:
    tok = SCEN_TOKEN[sched]
    for tag in ("p05", "p50", "p95"):
        case = f"smr100_{SHORT[tok]}_{tag}"
        idx = SELECTED[(sched, tag)]["idx"]
        w1 = WORLDS[sched].iloc[[idx]].reset_index(drop=True)
        CASES[case] = build_case_rec(case, sched, tag, w1,
                                     SELECTED[(sched, tag)]["score"], {"draw_index": idx})

selected_cases = pd.DataFrame({c: {k: v for k, v in rec.items() if not isinstance(v, np.ndarray)}
                               for c, rec in CASES.items()}).T.set_index("case")
selected_cases.to_csv(EXPORTS / "selected_cases.csv")
print(f"built {len(CASES)} production cases (6 schedules x p05/p50/p95); "
      "full record frozen to exports/smr100/selected_cases.csv")
selected_cases[["schedule", "case_type", "draw_index", "pctile_in_MC", "lr_smr", "boak_smr",
                "n_vendors", "convention", "occ_smr_2050", "occ_large_2050"]]

built 18 production cases (6 schedules x p05/p50/p95); full record frozen to exports/smr100/selected_cases.csv


,schedule,case_type,draw_index,pctile_in_MC,lr_smr,boak_smr,n_vendors,convention,occ_smr_2050,occ_large_2050
case,,,,,,,,,,
smr100_eia_p05,EIA 2026 AEO high,p05,2329,5.0,0.158923,6417.436975,7,tiny-base,3119.0,3837.0
smr100_eia_p50,EIA 2026 AEO high,p50,9573,50.0,0.080338,7565.05389,4,full-stock,5609.0,6357.0
smr100_eia_p95,EIA 2026 AEO high,p95,6874,95.0,0.126522,9732.341093,4,full-stock,7634.0,7374.0
smr100_aj_p05,Abou-Jaoude mod,p05,6077,5.0,0.144374,6457.191383,4,tiny-base,2862.0,4332.0
smr100_aj_p50,Abou-Jaoude mod,p50,242,50.0,0.04183,7048.38307,6,full-stock,5870.0,6107.0
smr100_aj_p95,Abou-Jaoude mod,p95,162,95.0,0.053308,9689.143019,6,full-stock,7666.0,7570.0
smr100_iaea_p05,IAEA high,p05,8423,5.0,0.09802,5550.338622,4,tiny-base,2839.0,4968.0
smr100_iaea_p50,IAEA high,p50,4993,50.0,0.127165,9263.333245,4,tiny-base,3766.0,5047.0
smr100_iaea_p95,IAEA high,p95,7602,95.0,0.032521,9137.897016,7,tiny-base,7616.0,7250.0


In [15]:
# --- The ITC-arm world records, selected from the materialized draws by u80 index ---
CASES_RD, RUN2CASE = {}, {}
for _, r in live.iterrows():
    ab, idx, case = r["schedule"], int(r["draw_index"]), r["case"]
    RUN2CASE[r["run"]] = case
    if case in CASES_RD:
        continue
    sched = STEM2SCHED[ab]
    w1 = WORLDS[sched].iloc[[idx]].reset_index(drop=True)
    CASES_RD[case] = build_case_rec(case, sched, "rd", w1,
                                    float(mc_score[sched][idx]), {"draw_index": idx},
                                    program="smr", score_dist=mc_score[sched])
assert len(CASES_RD) == N_LIVE

# --- The anchor world records (v2.2): built exactly as the smr100 p05/p50/p95 cases were
# (tag = percentile label, default program/score_dist), so their files are the smr100
# pattern with a new percentile label ---
CASES_ANCHOR = {}
for _, r in u82.iterrows():
    sched, idx, case = STEM2SCHED[r["schedule"]], int(r["draw_index"]), r["run"]
    w1 = WORLDS[sched].iloc[[idx]].reset_index(drop=True)
    CASES_ANCHOR[case] = build_case_rec(case, sched, r["percentile"], w1,
                                        float(mc_score[sched][idx]), {"draw_index": idx})
assert len(CASES_ANCHOR) == N_ANCHOR
assert not (set(CASES_ANCHOR) & set(CASES_RD))

# registration export: one row per world, all draw parameters + placement
sel_rows = []
for case, rec in {**CASES_RD, **CASES_ANCHOR}.items():
    sched, idx = rec["schedule"], rec["draw_index"]
    sel_rows.append({"case": case, "schedule": sched, "draw_index": idx,
                     "score": rec["score"],
                     "pctile_in_MC": round(100 * float(
                         (mc_score[sched] < rec["score"]).mean()), 2),
                     **{c: float(WORLDS[sched].iloc[idx][c])
                        for c in WORLDS[sched].columns}})
sel_df_rd = pd.DataFrame(sel_rows).set_index("case")
sel_df_rd.to_csv(EXPORTS / "selected_draws_ratedesign.csv")

# draw-identity assert vs mc_perdraw.npz for every selected world (ITC-arm + anchors)
ALL_CASES = {**CASES_RD, **CASES_ANCHOR}   # the ported Export 1/2 cells iterate ALL_CASES
if _npz_sel_path.exists():
    _npz_rd = np.load(_npz_sel_path, allow_pickle=False)
    _rd_cols = list(_npz_rd["world_columns"].astype(str))
    _rd_checked = 0
    for case, rec in ALL_CASES.items():
        tok = SCEN_TOKEN[rec["schedule"]]
        if _npz_rd[f"worlds_{tok}"].shape[0] != N_DRAWS:
            break
        assert _rd_cols == list(WORLDS[rec["schedule"]].columns)
        assert np.allclose(_npz_rd[f"worlds_{tok}"][rec["draw_index"]],
                           WORLDS[rec["schedule"]].iloc[rec["draw_index"]]
                           .to_numpy(dtype=float), rtol=1e-9, atol=0), case
        _rd_checked += 1
    if _rd_checked:
        print(f"draw identity vs mc_perdraw.npz: {_rd_checked}/{len(ALL_CASES)} worlds match")

print(f"built {len(CASES_RD)} ITC-arm + {len(CASES_ANCHOR)} anchor world records; "
      "input files follow")


draw identity vs mc_perdraw.npz: 60/60 worlds match
built 48 ITC-arm + 12 anchor world records; input files follow


## Exports to ReEDS: per-world input files

The Export 1/2 cells are ported verbatim; with `ALL_CASES` bound to the 48 ITC-arm +
12 anchor worlds they write 120 plantchar files, 60 `financials_tech_mc_*`, 60
`construction_times_mc_*` (`rd_*` for the ITC-arm worlds, `smr100_*_p25/_p75` for the
anchors), the idempotent `dollaryear.csv` registration, and re-write the shared
`construction_schedules_mc.csv` byte-identically (guard-verified in QA-R7).


In [16]:
# Export 1: plant-characteristics files + dollaryear registration
PLANTCHAR_DIR = REPO_ROOT / "inputs" / "plant_characteristics"
REEDS_TECH_NAME = {"large": "Nuclear", "smr": "Nuclear-SMR"}
ATB_FILE = {"large": "nuclear_ATB_2024_moderate.csv", "smr": "nuclear-smr_ATB_2024_moderate.csv"}
atb_base = {t: pd.read_csv(PLANTCHAR_DIR / ATB_FILE[t]) for t in TECH}

def plantchar_name(tech, case):
    return f"{'nuclear' if tech == 'large' else 'nuclear-smr'}_mc_{case}"

written_plantchar = []
for case, info in ALL_CASES.items():
    for tech in TECH:
        occ = info[f"occ_{tech}"]                             # (T,) 2024-2050, 2022 $/kW
        fom_d, vom_d = info[f"fom_{tech}"], info[f"vom_{tech}"]
        df = atb_base[tech].copy()
        for t_i, y in enumerate(YEARS):
            if y >= ANCHOR:                                   # pre-anchor years keep ATB values
                df.loc[df["t"] == y, "capcost"] = round(float(occ[t_i]), 1)
                df.loc[df["t"] == y, "fom"] = fom_d   # case-consistent O&M (2026-08-03)
                df.loc[df["t"] == y, "vom"] = vom_d
        name = plantchar_name(tech, case)
        df.to_csv(PLANTCHAR_DIR / f"{name}.csv", index=False)
        written_plantchar.append(name)

doll_path = PLANTCHAR_DIR / "dollaryear.csv"
doll = pd.read_csv(doll_path)
new_rows = [{"Scenario": n, "Dollar.Year": 2022} for n in written_plantchar
            if n not in set(doll["Scenario"])]
if new_rows:
    doll = pd.concat([doll, pd.DataFrame(new_rows)], ignore_index=True)
    doll.to_csv(doll_path, index=False)
print(f"wrote {len(written_plantchar)} plant-characteristics files; "
      f"registered {len(new_rows)} new dollaryear rows (idempotent)")

wrote 120 plant-characteristics files; registered 2 new dollaryear rows (idempotent)


In [17]:
# Export 2: shared construction-schedules columns + per-case financials_tech / construction_times
cs_mc = pd.read_csv(FIN_DIR / "construction_schedules_default.csv")
for tech, prefix in [("large", "NL"), ("smr", "NS")]:
    for n in range(1, 11):
        x = _resample_schedule(FIN["sched"][tech], n)
        col = np.zeros(len(cs_mc))
        col[1:n+1] = x                        # row 0 is the 'NA' (exponent-0) row
        cs_mc[f"{prefix}{n}"] = np.round(col, 6)
cs_mc.to_csv(FIN_DIR / "construction_schedules_mc.csv", index=False)

def dur_to_years(dur_row):
    """Designed duration path (months, (T,)) -> ReEDS spend-years labels per year."""
    return np.clip(np.round(np.asarray(dur_row) / 12.0).astype(int), 1, 10)

ft_base = pd.read_csv(FIN_DIR / "financials_tech_ATB2024.csv")
ct_base = pd.read_csv(FIN_DIR / "construction_times_default.csv")
SCH_PREFIX = {"large": "NL", "smr": "NS"}

for case, info in ALL_CASES.items():
    ft = ft_base.copy()
    ft["construction_sch"] = ft["construction_sch"].astype(str)
    ct = ct_base.copy()
    for tech in TECH:
        n_years = dur_to_years(info[f"dur_{tech}"])                     # (T,)
        iname = REEDS_TECH_NAME[tech]
        for t_i, y in enumerate(YEARS):
            if y >= ANCHOR:
                mask = (ft["i"] == iname) & (ft["t"] == y)
                ft.loc[mask, "construction_sch"] = f"{SCH_PREFIX[tech]}{n_years[t_i]}"
                ct.loc[(ct["i"] == iname) & (ct["t_online"] == y),
                       "construction_time"] = int(n_years[t_i])
    ft.to_csv(FIN_DIR / f"financials_tech_mc_{case}.csv", index=False)
    ct.to_csv(FIN_DIR / f"construction_times_mc_{case}.csv", index=False)
print(f"wrote construction_schedules_mc.csv (+20 profile columns) and "
      f"{len(ALL_CASES)} financials_tech_mc_* + {len(ALL_CASES)} construction_times_mc_* files")

wrote construction_schedules_mc.csv (+20 profile columns) and 60 financials_tech_mc_* + 60 construction_times_mc_* files


In [18]:
# --- Export 2b: per-world foreign-experience files at each world's own drawn u ---
# The generator's documented arbitrary-u mode (itcfb spec section 5); exact mode via the
# PRIS loader. Skips a file whose tag already exists with the same u (recorded in-file as
# a comment by the generator? no — recorded here in the registration export instead).
import subprocess, sys

_gen = REPO_ROOT / "inputs" / "nuclear_learning" / "_generate_foreign_experience.py"
_rds2 = REPO_ROOT / "z-ethan" / "mc" / "rds2_2025_units.csv"
assert _gen.exists() and _rds2.exists()

FOREIGN_TAG = {case: case for case in CASES_RD}      # foreign_experience_rd_{ab}_d{idx}.csv
for case, rec in CASES_RD.items():
    u_val = float(WORLDS[rec["schedule"]].iloc[rec["draw_index"]]["u"])
    out_f = REPO_ROOT / "inputs" / "nuclear_learning" / f"foreign_experience_{case}.csv"
    res = subprocess.run(
        [sys.executable, str(_gen), "--u", f"{u_val:.6f}", "--tag", case,
         "--rds2", str(_rds2)],
        capture_output=True, text=True, cwd=str(REPO_ROOT))
    assert res.returncode == 0, (case, res.stdout[-2000:], res.stderr[-2000:])
    assert out_f.exists(), case
    fx = pd.read_csv(out_f)
    assert list(fx.columns)[0] == "t" and fx["t"].max() == 2050
    assert fx["foreign_units_cum"].is_monotonic_increasing
print(f"wrote {len(CASES_RD)} foreign_experience_rd_* files (exact mode, per-world u)")


wrote 48 foreign_experience_rd_* files (exact mode, per-world u)


## Per-run incentives files

Exact minus-probe template (`build_itcfb_minus.py`): the `obbba_nonuclearitc` baseline with
its 8 zeroed `NUCLEAR` rows removed and one `Nuclear-SMR` row appended per build year —
`safe_harbor=0`, `t_max_online=t`, penalty 0.1, no bonuses, `itc_frac` = the u81
`rate_on_world` at 3 decimals. The file stores the headline rate; ReEDS monetizes at
×0.9 internally (`financials.py`). The hybrid runs' post-window cap (0.60; 0.50 for the
v2.2 `cb50` probes) is already baked into u81's offers, so no run needs special-casing
here. Anchor runs carry no incentives file (the no-nuclear-ITC baseline).


In [19]:
FIN_DIR_ = REPO_ROOT / "inputs" / "financials"
base_inc = (FIN_DIR_ / "incentives_obbba_nonuclearitc.csv").read_text(encoding="utf-8")
base_lines = [ln for ln in base_inc.splitlines() if ln.strip()]
kept = [ln for ln in base_lines if not ln.startswith("NUCLEAR,")]
assert len(base_lines) - len(kept) == 8
PEN = 0.1

def itc_row(tech, t, frac):
    return (f"{tech},usa,{t},0,{t},0.0,0.0,0,0.0,{frac:.3f},0.0,0.0,{PEN},"
            f"0.0,0,0.0,0.0,0,0.0")

INC_SUFFIX = {}
for run, grp in u81.groupby("run", sort=False):
    grp = grp.sort_values("year")
    fed = dict(zip(grp["year"].astype(int), grp["rate_on_world"].astype(float)))
    assert 0.0 < min(fed.values()) and max(fed.values()) < 1.0, run
    rows = [itc_row("Nuclear-SMR", t, round(f, 3)) for t, f in fed.items()]
    suffix = f"obbba_rd_{run}"
    INC_SUFFIX[run] = suffix
    (FIN_DIR_ / f"incentives_{suffix}.csv").write_text(
        "\n".join(kept + rows) + "\n", encoding="utf-8")
assert len(INC_SUFFIX) == N_LIVE
print(f"wrote {len(INC_SUFFIX)} incentives files "
      f"(headline-rate convention, x0.9 monetized in-model)")


wrote 48 incentives files (headline-rate convention, x0.9 monetized in-model)


## Horizon plumbing: 2055 extension

Registered configuration (methods.md v2.1): end year extended to 2055, mandate held flat
at its 2050 level, monetized-parity convention as in the feed-back runs. Implementation:

* **Extended mandate trajectories** `nuclear_cap_trajectory_{token}_smr_ext.csv` — the
  standard file's lines verbatim plus rows 2051–2055 repeating the 2050 MW value. Explicit
  files rather than a forecast.py fit: flat-at-2050 IS the registered design, and this
  keeps forecast.py away from the GAMS `*t` comment header.
* **futurefiles.csv ignore-rows** for the inputs_case files forecast.py would otherwise
  raise on (its missing-file check is a hard error): the five nuclear-learning files
  (trajectory pre-extended; the others historical/year-invariant — and the learning
  engine already clamps the foreign stock flat past its last year via `np.interp`), plus
  `wst_surface.csv` and the three run-root utility files, all uncovered in the shipped
  file. Idempotent append; the QA-R7 simulation asserts full coverage.


In [20]:
_nl_dir = REPO_ROOT / "inputs" / "nuclear_learning"
EXT_TOKEN = {}
for _, r in horiz.iterrows():
    tok = SCEN_TOKEN[STEM2SCHED[r["schedule"]]]
    src = _nl_dir / f"nuclear_cap_trajectory_{tok}_smr.csv"
    dst = _nl_dir / f"nuclear_cap_trajectory_{tok}_smr_ext.csv"
    lines = src.read_text(encoding="utf-8").rstrip("\n").split("\n")
    assert lines[0].startswith("*t"), src
    last_y, last_mw = lines[-1].split(",")
    assert int(last_y) == 2050, src
    ext = lines + [f"{y},{last_mw}" for y in range(2051, 2056)]
    dst.write_text("\n".join(ext) + "\n", encoding="utf-8")
    EXT_TOKEN[r["run"]] = f"{tok}_smr_ext"
print(f"wrote {len(set(EXT_TOKEN.values()))} extended mandate trajectories "
      f"(2051-2055 flat at the 2050 level)")


wrote 6 extended mandate trajectories (2051-2055 flat at the 2050 level)


In [21]:
# --- futurefiles.csv ignore-rows (idempotent) ---
_ff_path = REPO_ROOT / "inputs" / "userinput" / "futurefiles.csv"
_ff_text = _ff_path.read_text(encoding="utf-8")
_NOTE = "rate_design horizon block 2026-09-02: leave alone (pre-extended or year-invariant)"
_NEW_FF = [
    ("nuclear_cap_trajectory.csv", ".csv"),
    ("nuclear_cap_mandate_techs.csv", ".csv"),
    ("foreign_experience.csv", ".csv"),
    ("historical_stock.csv", ".csv"),
    ("inl_duration_curves.csv", ".csv"),
    ("wst_surface.csv", ".csv"),
    ("gamslice.txt", ".txt"),
    ("Project.toml", ".toml"),
    ("runreeds.py", ".py"),
]
_added = []
for name, ftype in _NEW_FF:
    if f"\n{name}," in _ff_text or _ff_text.startswith(f"{name},"):
        continue
    _ff_text = _ff_text.rstrip("\n") + \
        f"\n{name},{ftype},1,None,9999,None,9999,0,constant,None,None,{_NOTE},\n"
    _added.append(name)
if _added:
    _ff_path.write_text(_ff_text, encoding="utf-8")
print(f"futurefiles.csv: added {len(_added)} ignore-rows {_added or '(none - already present)'}")


futurefiles.csv: added 0 ignore-rows (none - already present)


In [22]:
# --- cases.csv Choices registration for the rd_* foreign-experience tags (idempotent) ---
# ReEDS validates switch values against cases.csv's Choices column with an unanchored
# re.match (reeds/inputs.py:227-241); GSw_NuclearLearning_ForeignScen enumerates its
# allowed tags, so the itcfb batch registered its fb_*_p50 tags there and this batch
# registers one regex token covering every rd_{sched}_d{draw} tag (any count). Comma-free (the
# cell is unquoted CSV); text-level edit so the rest of cases.csv is untouched.
_cases_path = REPO_ROOT / "cases.csv"
_cases_text = _cases_path.read_text(encoding="utf-8")
_RD_CHOICE = r"rd_(eia|aj|iaea|mck|cop28|eo)_d\d+"
if _RD_CHOICE not in _cases_text:
    _anchor = "fb_eo_p50,mid,"
    assert _cases_text.count(_anchor) == 1, "cases.csv ForeignScen row drifted"
    _cases_text = _cases_text.replace(_anchor, f"fb_eo_p50; {_RD_CHOICE},mid,")
    _cases_path.write_text(_cases_text, encoding="utf-8")
    print(f"cases.csv: registered Choices token {_RD_CHOICE!r} for "
          "GSw_NuclearLearning_ForeignScen")
else:
    print("cases.csv: rd_* Choices token already registered")


cases.csv: rd_* Choices token already registered


## Export 3: the cases file

66 columns on the itcfbm 31-row template. Live runs = the fbC pattern with per-world
values: no mandate, per-run incentives, learning ON with the world's own drawn parameters,
per-world file pointers. Anchor runs (v2.2) = the `smr100_{sched}_p50` anchor columns
verbatim (mandate ON, no-nuclear-ITC default, learning OFF, endyear 2050) with their own
four file pointers. Horizon runs = the same anchor columns except `endyear=2055`, the
extended yearset, and the `_ext` mandate token. Column order: live → anchor → horizon.


In [23]:
tmpl = pd.read_csv(REPO_ROOT / "cases_nuclearlearning_itcfbm.csv",
                   index_col=0, dtype=str).fillna("")
base_s100 = pd.read_csv(REPO_ROOT / "cases_nuclearlearning_smr100.csv",
                        index_col=0, dtype=str).fillna("")
ROWS = list(tmpl.index)
assert len(ROWS) == 31 and ROWS[0] == "ignore"

COPY_ROWS = ["GSw_NuclearCapMandateScen", "plantchar_nuclear", "plantchar_nuclear_smr",
             "financials_tech_suffix", "construction_times_suffix"]
LEARN_FMT = {
    "GSw_NuclearLearning_LR_large":      ("lr_large",   lambda v: f"{v:.6f}"),
    "GSw_NuclearLearning_LR_smr":        ("lr_smr",     lambda v: f"{v:.6f}"),
    "GSw_NuclearLearning_BOAK_large":    ("boak_large", lambda v: f"{v:.2f}"),
    "GSw_NuclearLearning_BOAK_smr":      ("boak_smr",   lambda v: f"{v:.2f}"),
    "GSw_NuclearLearning_Vendors":       ("n_vendors",  lambda v: str(int(v))),
    "GSw_NuclearLearning_Convention":    ("conv_full",  lambda v: str(int(v))),
    "GSw_NuclearLearning_CES_rho":       ("ces_rho",    lambda v: f"{v:g}"),
    "GSw_NuclearLearning_Spillover":     ("s",          lambda v: f"{v:.6f}"),
    "GSw_NuclearLearning_CrossTech_x_ls": ("x_ls",      lambda v: f"{v:.6f}"),
    "GSw_NuclearLearning_CrossTech_x_sl": ("x_sl",      lambda v: f"{v:.6f}"),
    "GSw_NuclearLearning_Dur_Lambda":    ("dur_lambda", lambda v: f"{v:.6f}"),
}
YEARSET_STD = tmpl.loc["yearset", "Default Value"]
YEARSET_EXT = YEARSET_STD + "_2053_2055"

cols = {}
for _, r in live.iterrows():
    case = RUN2CASE[r["run"]]
    rec = CASES_RD[case]
    w = WORLDS[rec["schedule"]].iloc[rec["draw_index"]]
    c = {row: "" for row in ROWS}
    c["GSw_NuclearCapMandate"] = "0"
    c["incentives_suffix"] = INC_SUFFIX[r["run"]]
    c["GSw_NuclearCapMandateScen"] = SCEN_TOKEN[rec["schedule"]] + "_smr"
    c["plantchar_nuclear"] = plantchar_name("large", case)
    c["plantchar_nuclear_smr"] = plantchar_name("smr", case)
    c["financials_tech_suffix"] = f"mc_{case}"
    c["construction_times_suffix"] = f"mc_{case}"
    c["GSw_NuclearLearning"] = "1"
    c["GSw_NuclearLearning_OCC"] = "1"
    c["GSw_NuclearLearning_Duration"] = "1"
    c["GSw_NuclearLearning_CrossTech"] = "1"
    for sw, (dcol, fmt) in LEARN_FMT.items():
        c[sw] = fmt(float(w[dcol]))
    c["GSw_NuclearLearning_ForeignScen"] = case
    cols[r["run"]] = c

# anchor block (v2.2): the smr100 anchor column pattern with the world's own file pointers;
# the mandate token is the schedule's standard {tok}_smr (copied from the p50 column)
for _, r in u82.iterrows():
    stem, case = r["schedule"], r["run"]
    src_case = f"smr100_{stem}_p50"
    c = {row: "" for row in ROWS}
    for row in COPY_ROWS:
        c[row] = base_s100.loc[row, src_case]
    c["plantchar_nuclear"] = plantchar_name("large", case)
    c["plantchar_nuclear_smr"] = plantchar_name("smr", case)
    c["financials_tech_suffix"] = f"mc_{case}"
    c["construction_times_suffix"] = f"mc_{case}"
    c["GSw_TCPhaseout_NuclearExempt"] = "0"     # resolved-switch parity with the anchors
    cols[case] = c

for _, r in horiz.iterrows():
    stem = r["schedule"]
    src_case = f"smr100_{stem}_p50"
    c = {row: "" for row in ROWS}
    for row in COPY_ROWS:
        c[row] = base_s100.loc[row, src_case]
    c["GSw_NuclearCapMandateScen"] = EXT_TOKEN[r["run"]]
    c["endyear"] = "2055"
    c["yearset"] = YEARSET_EXT
    # resolved-switch parity with the anchors (registered 09-02): the itcfbm template's
    # Default Value pins GSw_TCPhaseout_NuclearExempt=1 (the fb runs need it), but the
    # smr100 anchors ran the cases.csv default 0 — pin the horizon cells to 0 explicitly.
    c["GSw_TCPhaseout_NuclearExempt"] = "0"
    cols[r["run"]] = c

N_COLS = N_LIVE + N_ANCHOR + N_HZ
assert len(cols) == N_COLS == 66, (len(cols), N_COLS)
defaults = {row: tmpl.loc[row, "Default Value"] for row in ROWS}
cases_rd = pd.DataFrame({"Default Value": defaults} | cols).reindex(ROWS)
cases_rd.index.name = ""
RD_CASES_PATH = REPO_ROOT / "cases_nuclearlearning_ratedesign.csv"
cases_rd.to_csv(RD_CASES_PATH)
print(f"wrote {RD_CASES_PATH.name}: {cases_rd.shape[0]} switch rows x "
      f"{cases_rd.shape[1] - 1} run columns "
      f"(launch: python runreeds.py -b <batch> -c nuclearlearning_ratedesign)")


wrote cases_nuclearlearning_ratedesign.csv: 31 switch rows x 66 run columns (launch: python runreeds.py -b <batch> -c nuclearlearning_ratedesign)


## QA suite

Ported world-identity gate (QA-0) plus the transcription gates: QA-R1 casefile ↔ u80,
QA-R2 incentives ↔ u81 + baseline byte-identity, QA-R3 rate bounds, QA-R4 per-world
switch identity, QA-R5 plantchar ↔ u81 occ round-trip (closes the loop between the
rate_design offline machinery and the emitted run inputs), QA-R6 horizon columns +
extended trajectories + yearset, QA-R7 futurefiles coverage simulation + cases.csv
validation + frozen-artifact byte-identity, QA-R8 (v2.2) anchor columns: resolved-switch
parity with the p50 anchor of the same schedule except the four file pointers, pointer
files present, quantile placement, distinctness.


In [24]:
# QA-0 — same sample space as the companion: if mc_cost_trajectories.ipynb's per-draw export
# (exports/mc_perdraw.npz) is present and was generated at the same draw count, this
# notebook's drawn worlds must match it draw-for-draw (same code, same seed streams), and the
# SMR full-program trajectory arrays must match too. Tolerance is floating-point rounding
# only (rtol 1e-9): the companion runs on a different kernel/BLAS build, which perturbs the
# copula matmul in the last bit (~2e-16 relative) — the underlying random stream is identical
# (the plain-rng columns like `s` match bit-for-bit). Skips gracefully if absent/stale.
_npz_path = NB_DIR / "exports" / "mc_perdraw.npz"
if _npz_path.exists():
    _pd_npz = np.load(_npz_path, allow_pickle=False)
    _tok0 = SCEN_TOKEN[SCHED_ORDER[0]]
    if _pd_npz[f"worlds_{_tok0}"].shape[0] != N_DRAWS:
        print(f"QA-0 SKIPPED: mc_perdraw.npz has {_pd_npz[f'worlds_{_tok0}'].shape[0]} draws, "
              f"this run has {N_DRAWS} — re-run the companion notebook to compare")
    else:
        cols = list(_pd_npz["world_columns"].astype(str))
        for sched in SCHED_ORDER:
            tok = SCEN_TOKEN[sched]
            assert cols == list(WORLDS[sched].columns), "world column layout diverged"
            ours_w = WORLDS[sched].to_numpy(dtype=float)
            assert np.allclose(_pd_npz[f"worlds_{tok}"], ours_w, rtol=1e-9, atol=0), tok
            assert np.array_equal(_pd_npz[f"worlds_{tok}"][:, cols.index("s")],
                                  WORLDS[sched]["s"].to_numpy()), (tok, "stream identity")
            for ch, ours in (("occ", "occ_smr"), ("dur", "dur_smr"),
                             ("ccmult", "ccmult_smr"), ("fincapex", "fincapex_smr")):
                assert np.allclose(_pd_npz[f"{ch}_{tok}_smr"], results[sched][ours],
                                   rtol=1e-9, atol=0), (tok, ch)
        print("QA-0 PASSED: drawn worlds and SMR full-program trajectories match the companion "
              "notebook's mc_perdraw.npz draw-for-draw (bit-identical random stream; "
              "differences bounded by cross-kernel floating-point rounding) — "
              "same sample space by construction")
else:
    print("QA-0 SKIPPED: exports/mc_perdraw.npz not found (run the companion notebook to enable "
          "the sample-space cross-check)")

QA-0 PASSED: drawn worlds and SMR full-program trajectories match the companion notebook's mc_perdraw.npz draw-for-draw (bit-identical random stream; differences bounded by cross-kernel floating-point rounding) — same sample space by construction


In [25]:
# QA-R1 -- casefile transcription of u80 + u82: names, order, horizon last, N_COLS columns.
_back = pd.read_csv(RD_CASES_PATH, index_col=0, dtype=str).fillna("")
_expected = list(live["run"]) + list(u82["run"]) + list(horiz["run"])
assert list(_back.columns) == ["Default Value"] + _expected
assert [c for c in _back.columns if c.startswith("hz_")] == list(_back.columns[-N_HZ:])
assert list(_back.columns[1 + N_LIVE:1 + N_LIVE + N_ANCHOR]) == list(u82["run"])
assert list(_back.index) == ROWS
assert not any(c.startswith("reserve") for c in _back.columns)   # v2.2: no reserve
print(f"QA-R1 PASSED: {N_COLS} columns in u80/u82 order (envelope, boundary, hybrid, "
      f"anchor, horizon LAST); {len(ROWS)} template rows; no reserve")


QA-R1 PASSED: 66 columns in u80/u82 order (envelope, boundary, hybrid, anchor, horizon LAST); 31 template rows; no reserve


In [26]:
# QA-R2 -- incentives echo vs u81 + baseline byte-identity.
for run in list(live["run"]):
    txt = (FIN_DIR_ / f"incentives_{INC_SUFFIX[run]}.csv").read_text(encoding="utf-8")
    lines = [ln for ln in txt.splitlines() if ln.strip()]
    body = [ln for ln in lines if not ln.startswith("Nuclear-SMR,")]
    assert body == kept, run                      # non-nuclear rows byte-identical
    nuc = [ln.split(",") for ln in lines if ln.startswith("Nuclear-SMR,")]
    got = {int(p[2]): float(p[9]) for p in nuc}
    grp = u81[u81["run"] == run]
    want = {int(t): round(float(x), 3)
            for t, x in zip(grp["year"], grp["rate_on_world"])}
    assert got == want, (run, got, want)
    for p in nuc:                                  # row shape: the minus-probe template
        assert p[3] == "0" and p[4] == p[2] and p[12] == "0.1", run
        assert p[10] == "0.0" and p[11] == "0.0", run
print(f"QA-R2 PASSED: {N_LIVE} incentives files echo u81 exactly (rate years, 3-dp headline "
      "rates, safe_harbor 0, penalty 0.1, no bonuses); baselines byte-identical")


QA-R2 PASSED: 48 incentives files echo u81 exactly (rate years, 3-dp headline rates, safe_harbor 0, penalty 0.1, no bonuses); baselines byte-identical


In [27]:
# QA-R3 -- rate bounds: every fed rate < 1.0; per-run max matches u80's registered max.
for _, r in live.iterrows():
    grp = u81[u81["run"] == r["run"]]
    mx = round(float(grp["rate_on_world"].max()), 3)
    assert mx < 1.0
    assert abs(mx - float(r["max_rate_on_world"])) <= 0.0011, (r["run"], mx)
print("QA-R3 PASSED: all offered rates < 1.0; per-run maxima match u80")


QA-R3 PASSED: all offered rates < 1.0; per-run maxima match u80


In [28]:
# QA-R4 -- per-world switch identity: the casefile's learning switches reproduce the
# drawn world exactly (via LEARN_FMT), and the foreign tag matches the emitted file.
for _, r in live.iterrows():
    rec = CASES_RD[RUN2CASE[r["run"]]]
    w = WORLDS[rec["schedule"]].iloc[rec["draw_index"]]
    for sw, (dcol, fmt) in LEARN_FMT.items():
        assert _back.loc[sw, r["run"]] == fmt(float(w[dcol])), (r["run"], sw)
    tag = _back.loc["GSw_NuclearLearning_ForeignScen", r["run"]]
    assert (REPO_ROOT / "inputs" / "nuclear_learning"
            / f"foreign_experience_{tag}.csv").exists(), r["run"]
    assert _back.loc["GSw_NuclearLearning", r["run"]] == "1"
    assert _back.loc["GSw_NuclearCapMandate", r["run"]] == "0"
print(f"QA-R4 PASSED: all {N_LIVE} live columns carry their world's exact drawn parameters, "
      "fbC learning flags, no mandate, and an existing foreign-experience file")


QA-R4 PASSED: all 48 live columns carry their world's exact drawn parameters, fbC learning flags, no mandate, and an existing foreign-experience file


In [29]:
# QA-R5 -- plantchar <-> u81 round-trip: the occ path in the written plantchar file
# (native 2022 $/kW) equals u81's occ_world_kW after the SAME 2022->2024 conversion the
# rate_design builder registers (D2224 = infl_2023 * infl_2024 from deflator.csv), and
# offer/occ reproduces rate_on_world (rates are unitless, so the incentives files are
# dollar-year-invariant). This closes the loop between the rate_design offline machinery
# and the run inputs ReEDS will actually see.
PLANTCHAR_DIR_ = REPO_ROOT / "inputs" / "plant_characteristics"
_infl_rd = pd.read_csv(REPO_ROOT / "inputs" / "financials" / "inflation_default.csv")
_infl_rd = _infl_rd.set_index("t")["inflation_rate"]
D2224 = float(_infl_rd.loc[2023] * _infl_rd.loc[2024])
assert 1.0 < D2224 < 1.2, D2224
for _, r in live.iterrows():
    case = RUN2CASE[r["run"]]
    back = pd.read_csv(PLANTCHAR_DIR_ / f"{plantchar_name('smr', case)}.csv")
    back = back.set_index("t")["capcost"]
    grp = u81[u81["run"] == r["run"]]
    for _, q in grp.iterrows():
        occ_file = float(back.loc[int(q["year"])])            # 2022 $/kW (native)
        occ_2024 = occ_file * D2224
        assert abs(occ_2024 - float(q["occ_world_kW"])) <= 0.06 + 1e-4 * occ_2024, \
            (r["run"], q["year"], occ_2024, float(q["occ_world_kW"]))
        assert abs(float(q["offer_kW"]) / occ_2024
                   - float(q["rate_on_world"])) <= 5e-4, (r["run"], q["year"])
print(f"QA-R5 PASSED: every run's plantchar occ path x D2224 ({D2224:.4f}) equals u81's "
      "2024$ world path, and offer/occ reproduces the registered rate (<= 5e-4)")


QA-R5 PASSED: every run's plantchar occ path x D2224 (1.0712) equals u81's 2024$ world path, and offer/occ reproduces the registered rate (<= 5e-4)


In [30]:
# QA-R6 -- horizon columns: RESOLVED-switch-verbatim vs the smr100 p50 anchors except the
# three registered fields; extended trajectories correct; endyear in the extended yearset.
# Resolution follows reeds/inputs.py:169-185: case cell -> casefile Default Value ->
# cases.csv Default Value. (Amended 09-02 after external review: the raw-cell comparison
# was blind to Default-column divergence — GSw_TCPhaseout_NuclearExempt resolved to 1 here
# vs 0 in the anchors; the six horizon cells now pin it to 0 explicitly.)
_cm6 = pd.read_csv(REPO_ROOT / "cases.csv", index_col=0)
_cm6_def = {str(k): ("" if pd.isna(v) else str(v))
            for k, v in _cm6["Default Value"].items()}

def _resolved(df, col, row):
    v = str(df.loc[row, col]) if row in df.index else ""
    if v == "":
        v = str(df.loc[row, "Default Value"]) if row in df.index else ""
    if v == "":
        v = _cm6_def.get(row, "")
    return v

_EXEMPT6 = {"endyear", "yearset", "GSw_NuclearCapMandateScen", "ignore"}
for _, r in horiz.iterrows():
    run, stem = r["run"], r["schedule"]
    src_case = f"smr100_{stem}_p50"
    assert _back.loc["endyear", run] == "2055", run
    assert _back.loc["yearset", run] == YEARSET_EXT, run
    assert _back.loc["GSw_NuclearCapMandateScen", run] == EXT_TOKEN[run] == \
        base_s100.loc["GSw_NuclearCapMandateScen", src_case] + "_ext", run
    for row in ROWS:
        if row in _EXEMPT6:
            continue
        got = _resolved(_back, run, row)
        want = _resolved(base_s100, src_case, row)
        assert got == want, (run, row, got, want)
    tok = SCEN_TOKEN[STEM2SCHED[stem]]
    std = (_nl_dir / f"nuclear_cap_trajectory_{tok}_smr.csv").read_text(encoding="utf-8")
    ext = (_nl_dir / f"nuclear_cap_trajectory_{tok}_smr_ext.csv").read_text(encoding="utf-8")
    std_l = std.rstrip("\n").split("\n")
    ext_l = ext.rstrip("\n").split("\n")
    assert ext_l[:len(std_l)] == std_l, tok                 # prefix byte-identical
    mw2050 = std_l[-1].split(",")[1]
    assert ext_l[len(std_l):] == [f"{y},{mw2050}" for y in range(2051, 2056)], tok
ys = [int(t) for t in YEARSET_EXT.split("_")]
assert 2055 in ys and 2053 in ys and ys == sorted(ys)
print("QA-R6 PASSED: horizon columns match the p50 anchors on RESOLVED switch values "
      "except endyear/yearset/mandate-scen; _ext trajectories flat at the 2050 MW value; "
      "endyear 2055 is a yearset member")


QA-R6 PASSED: horizon columns match the p50 anchors on RESOLVED switch values except endyear/yearset/mandate-scen; _ext trajectories flat at the 2050 MW value; endyear 2055 is a yearset member


In [31]:
# QA-R8 (v2.2) -- anchor columns: RESOLVED-switch parity with the smr100 p50 anchor of the
# same schedule except the four per-world file pointers; pointer files exist and carry the
# world's own occ path; quantile placement by the frozen rule; distinct from the frozen 18
# and from every ITC-arm world; no incentives/foreign files (baseline, learning OFF).
_EXEMPT8 = {"plantchar_nuclear", "plantchar_nuclear_smr", "financials_tech_suffix",
            "construction_times_suffix", "ignore"}
_frozen18_r8 = {(r["schedule"], int(r["draw_index"])) for _, r in _frozen_sel.iterrows()}
for _, r in u82.iterrows():
    run, stem = r["run"], r["schedule"]
    src_case = f"smr100_{stem}_p50"
    sched = STEM2SCHED[stem]
    for row in ROWS:
        if row in _EXEMPT8:
            continue
        got, want = _resolved(_back, run, row), _resolved(base_s100, src_case, row)
        assert got == want, (run, row, got, want)
    assert _resolved(_back, run, "GSw_NuclearLearning") == "0", run
    assert _resolved(_back, run, "GSw_NuclearCapMandate") == "1", run
    assert _resolved(_back, run, "endyear") == "2050", run
    assert _back.loc["plantchar_nuclear_smr", run] == plantchar_name("smr", run)
    assert _back.loc["plantchar_nuclear", run] == plantchar_name("large", run)
    for p in [PLANTCHAR_DIR_ / f"{plantchar_name('smr', run)}.csv",
              PLANTCHAR_DIR_ / f"{plantchar_name('large', run)}.csv",
              FIN_DIR_ / f"financials_tech_mc_{run}.csv",
              FIN_DIR_ / f"construction_times_mc_{run}.csv"]:
        assert p.exists(), (run, p.name)
    assert not (FIN_DIR_ / f"incentives_obbba_rd_{run}.csv").exists(), run
    # the written plantchar occ path is the world's own designed path (2022 $/kW, rounded)
    back8 = pd.read_csv(PLANTCHAR_DIR_ / f"{plantchar_name('smr', run)}.csv")
    back8 = back8.set_index("t")["capcost"]
    occ8 = CASES_ANCHOR[run]["occ_smr"]
    for t_i, y in enumerate(YEARS):
        if y >= ANCHOR:
            assert abs(float(back8.loc[y]) - round(float(occ8[t_i]), 1)) < 1e-6, (run, y)
    # quantile placement + distinctness
    idx = int(r["draw_index"])
    order8 = np.argsort(mc_score[sched], kind="stable")
    assert idx == int(order8[int(np.ceil(float(r["q"]) * (N_DRAWS - 1)))]), run
    assert (sched, idx) not in _frozen18_r8, run
    assert (stem, idx) not in set(live_worlds), run
print(f"QA-R8 PASSED: {N_ANCHOR} anchor columns match the p50 anchors on RESOLVED switch "
      "values except the four file pointers; pointer files present with the world's own "
      "occ path; frozen quantile rule reproduced; distinct from the 18 frozen anchors and "
      "all ITC-arm worlds; no incentives file")


QA-R8 PASSED: 12 anchor columns match the p50 anchors on RESOLVED switch values except the four file pointers; pointer files present with the world's own occ path; frozen quantile rule reproduced; distinct from the 18 frozen anchors and all ITC-arm worlds; no incentives file


In [32]:
# QA-R7 -- (a) futurefiles coverage simulation for the horizon configs, (b) cases.csv
# Choices validation of the new casefile, (c) frozen-artifact byte-identity.
import re as _re

# (a) simulate forecast.py's missing-file check: every runfiles.csv file required under
# the horizon switch config must have a futurefiles.csv row (forecast.py raises otherwise).
runfiles = pd.read_csv(REPO_ROOT / "reeds" / "input_processing" / "runfiles.csv",
                       comment="#")
ff_names = set(pd.read_csv(REPO_ROOT / "inputs" / "userinput" / "futurefiles.csv")
               ["filename"])
cases_master = pd.read_csv(REPO_ROOT / "cases.csv", index_col=0)

class _SW:
    def __init__(self, d): self.__dict__.update(d)

for run in [r["run"] for _, r in horiz.iterrows()]:
    swd = {k: str(cases_master.loc[k, "Default Value"])
           for k in cases_master.index if isinstance(k, str)}
    for row in ROWS:
        v = _back.loc[row, run] or _back.loc[row, "Default Value"]
        if v != "":
            swd[row] = str(v)
    sw = _SW(swd)
    missing = []
    for _, rf in runfiles.iterrows():
        name, req = str(rf["filename"]), str(rf["required_if"])
        if "." not in name or "{" in name:
            continue      # brace names are switch-formatted before landing; their
                          # inputs_case names are the runfiles 'filename' column, no braces
        try:
            required = bool(eval(req, {"int": int, "float": float, "str": str},
                                 {"sw": sw}))
        except Exception:
            required = True                      # conservative: assume it lands
        if required and name not in ff_names:
            missing.append(name)
    assert not missing, (run, missing)
print("QA-R7a PASSED: every inputs_case file implied by the horizon configs has a "
      "futurefiles.csv row (forecast.py's raise-on-missing check will pass)")

# (b) the new casefile validates against cases.csv's own index and Choices, replicating
# ReEDS's ACTUAL validation semantics (reeds/inputs.py:227-241): unanchored re.match,
# i.e. prefix matching — the same gate the itcfb/itcfbm suffixes passed through.
for switch in _back.index:
    if switch == "ignore":
        continue
    assert switch in cases_master.index, f"unknown switch {switch}"
    choices = str(cases_master.loc[switch, "Choices"])
    for col in _back.columns:
        val = _back.loc[switch, col]
        if val == "" or choices in ("N/A", "nan", "None"):
            continue
        if choices.lower() in ("int", "integer"):
            int(val)
            continue
        if choices.lower() in ("float", "numeric", "number", "num"):
            float(val)
            continue
        i_choices = [str(j).strip() for j in
                     np.ravel([c.split(",") for c in choices.split(";")]).tolist()]
        assert any(_re.match(ch, str(val)) for ch in i_choices), \
            (switch, val, choices)
print("QA-R7b PASSED: the casefile passes ReEDS's own Choices validation "
      "(reeds/inputs.py semantics, prefix re.match)")

# (c) frozen artifacts byte-identical (incl. the deterministically re-written
# construction_schedules_mc.csv)
for p, sha0 in GUARD_SHA.items():
    assert _sha(p) == sha0, f"frozen artifact changed: {p}"
print("QA-R7c PASSED: all snapshotted frozen artifacts byte-identical")


QA-R7a PASSED: every inputs_case file implied by the horizon configs has a futurefiles.csv row (forecast.py's raise-on-missing check will pass)
QA-R7b PASSED: the casefile passes ReEDS's own Choices validation (reeds/inputs.py semantics, prefix re.match)
QA-R7c PASSED: all snapshotted frozen artifacts byte-identical


## Run metadata

In [33]:
import json
import subprocess as _sp
from datetime import datetime, timezone

try:
    _git = _sp.run(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT,
                   capture_output=True, text=True).stdout.strip()
except Exception:
    _git = "unavailable"

meta = {
    "notebook": "ratedesign_case_export.ipynb",
    "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "fork_git_head": _git,
    "master_seed": MASTER_SEED,
    "built_by": "_build_ratedesign_case_export.py (ported cells verbatim from "
                "smr100_case_export.ipynb with build-time content asserts)",
    "design": {
        "batch": f"rate_design v2.2 pre-registered batch (2026-09-04), {N_COLS} launchable "
                 f"columns: {BLOCK_N['envelope']} envelope + {BLOCK_N['boundary']} "
                 f"boundary-depth + {BLOCK_N['hybrid']} hybrid (fbC convention: learning ON "
                 "with the run world's drawn parameters, no mandate, per-run credit; hybrid "
                 f"caps 0.60 and 0.50) + {N_ANCHOR} p25/p75 anchor-densification runs "
                 "(smr100 pattern: mandate ON, learning OFF, no-nuclear-ITC, endyear 2050) "
                 f"+ {N_HZ} horizon reruns of the smr100 p50 anchors (endyear 2055, mandate "
                 "flat at 2050 via _ext trajectories). No reserve (v2.2). "
                 "HORIZON BLOCK LAST (Ethan 09-02); anchor block before it.",
        "spec_source": "z-ethan/rate_design/exports/u80_batch_spec.csv + "
                       "u81_run_schedules.csv (byte-identity guarded) + u82_anchor_spec.csv "
                       "(selected here by the frozen smr100 quantile rule, methods.md v2.2)",
        "anchor_rule": "idx = argsort(rank_smr score, stable)[ceil(q*(N-1))], q = 0.25/0.75; "
                       "frozen selected_draws.csv untouched; registration in "
                       "exports/smr100/selected_draws_p25p75.csv",
        "arm_clarification": "envelope/boundary/hybrid = fbC (learning ON): the delivery "
                             "certificate the batch extends (fbC full-headline delivery + "
                             "the r03 bracket) was earned on that arm; logged in "
                             "rate_design/status.md as a build-time clarification",
        "incentives_convention": "headline rate in the file, x0.9 monetized in-model "
                                 "(the registered monetized-parity convention); "
                                 "minus-probe row template",
        "horizon": {"endyear": 2055, "yearset": "std + _2053_2055",
                    "mandate": "flat at 2050 via nuclear_cap_trajectory_{tok}_smr_ext",
                    "futurefiles": "ignore-rows added for the five nuclear-learning "
                                   "files + wst_surface + run-root utility files"},
        "copula_set": COPULA_SET, "n_draws": N_DRAWS},
    "runs": {r["run"]: {"block": r["block"], "schedule": r["schedule"],
                        "draw_index": int(r["draw_index"]),
                        "gate": r["gate"], "expected": r["expected"]}
             for _, r in pd.concat([u80, u82[["run", "block", "schedule", "draw_index",
                                               "gate", "expected"]]]).iterrows()},
    "reeds_files": {
        "cases_file": RD_CASES_PATH.name,
        "plantchar": sorted(written_plantchar),
        "financials_tech": [f"financials_tech_mc_{c}.csv" for c in ALL_CASES],
        "construction_times": [f"construction_times_mc_{c}.csv" for c in ALL_CASES],
        "foreign_experience": [f"foreign_experience_{c}.csv" for c in CASES_RD],
        "incentives": [f"incentives_{s}.csv" for s in INC_SUFFIX.values()],
        "mandate_trajectories_ext": sorted(set(EXT_TOKEN.values())),
        "construction_schedules": "construction_schedules_mc.csv (unchanged, "
                                  "byte-identity asserted)"},
    "exports": {p.name: _sha(p)[:16] for p in sorted(EXPORTS.glob("*.csv"))},
}
with open(EXPORTS / "ratedesign_metadata.json", "w") as f:
    json.dump(meta, f, indent=2)
print(json.dumps({k: meta[k] for k in ("notebook", "generated_utc", "fork_git_head")},
                 indent=2))
print(f"full metadata: {EXPORTS / 'ratedesign_metadata.json'}")


{
  "notebook": "ratedesign_case_export.ipynb",
  "generated_utc": "2026-09-04T19:15:49+00:00",
  "fork_git_head": "aa16beadfd6af29ea718a1875270e271fbb0e081"
}
full metadata: C:\Users\ethan\code\research\ReEDS-nuclear-learning\z-ethan\mc\exports\ratedesign\ratedesign_metadata.json


## Closing note

**One case file, 66 launchable runs (v2.2)**: `cases_nuclearlearning_ratedesign.csv`
(launch: `python runreeds.py -b <batch> -c nuclearlearning_ratedesign`). Column order:
48 ITC-arm runs (envelope, boundary, hybrid) → 12 p25/p75 anchors → the 6 horizon runs
deliberately LAST — launch in column order and the 60 standard-horizon runs return results
even if the never-before-exercised 2055 extension path fails in input processing. No
reserve (methods.md v2.2: one batch, every slot live).

New input files shipped with the batch: 120 plantchar, 60 financials_tech, 60
construction_times, 48 foreign_experience, 48 incentives, 6 extended mandate trajectories,
plus the (already present) 9 futurefiles.csv ignore-rows; new registrations
`u82_anchor_spec.csv` and `exports/smr100/selected_draws_p25p75.csv`. Nothing frozen was
touched (QA-R7c). Gates GE1/GE2/GH1/GH2/GX1 and GA1/GA2 are adjudicated when the batch
returns — nothing is drafted before then.
